# 15 — Constraint Error Trajectory Mechanisms

## Motivation

Notebook `14_trajectory_context_utility` established that trajectory context provides selective rather than universal utility.

The strongest effect was concentrated in `constraint_error`:

- support: 317
- semantic recall: 0.3817
- transition recall: 0.4416
- recall improvement: +0.0599
- transition rescues: 26
- transition breaks: 7
- net rescues: +19

By comparison, trajectory context was only mildly beneficial for `workflow_error` and harmful overall for `grounding_state_error`, `tool_use_error`, and `reasoning_value_error`.

Attempts to improve performance through increasingly sophisticated routing produced only modest gains. Raw trajectory features were also weak predictors of whether a transition override would be correct once the transition model's posterior was already available.

Therefore the next research question is no longer:

> Does trajectory context help?

Instead, this notebook asks:

> **What information in interaction history allows the transition representation to recognize constraint errors that the current-turn semantic representation misses?**

---

## Main research questions

### RQ1 — What distinguishes rescued constraint errors from non-rescued constraint errors?

Compare true `constraint_error` examples belonging to:

1. semantic wrong / transition correct — trajectory rescue
2. semantic correct / transition wrong — trajectory break
3. both correct
4. both wrong

The goal is to determine whether rescues have distinctive trajectory structure.

### RQ2 — How far back in history does the useful signal occur?

Determine whether constraint rescues depend primarily on:

- t-1
- t-2
- t-3
- longer history
- transition-to-transition similarity

This tests whether constraint failures are primarily local continuity failures or genuinely long-range state failures.

### RQ3 — What does trajectory context change in probability space?

For rescued constraint errors, analyze how adding trajectory information changes:

- constraint probability
- predicted class
- class rank
- margin
- entropy

The objective is to understand whether trajectory context:

- strongly introduces constraint evidence,
- weakly re-ranks an already plausible constraint prediction,
- or suppresses a competing semantic interpretation.

### RQ4 — Which semantic confusions are repaired?

Analyze:

true class = constraint_error  
semantic prediction != constraint_error  
transition prediction = constraint_error

Determine which semantic classes are most frequently corrected.

Previous results suggest important flows such as:

- workflow_error → constraint_error
- tool_use_error → constraint_error
- grounding_state_error → constraint_error

but these mechanisms should now be analyzed directly.

### RQ5 — Are there multiple kinds of trajectory-sensitive constraint errors?

Investigate whether the 26 rescued constraint examples form distinct subgroups.

Possible mechanisms may include:

- recent-history dependence
- long-history dependence
- abrupt state changes
- repeated-state consistency
- weak semantic ambiguity resolved by history
- accumulated constraints

These mechanisms should be inferred from the data rather than assumed in advance.

---

## Working hypothesis

Trajectory context is expected to be most useful when the current turn is locally compatible with several failure families but previous interaction state makes one interpretation inconsistent with an established constraint.

Under this hypothesis, trajectory context does not simply provide more semantic information.

Instead, it provides **state information** that changes how the current action should be interpreted.

This notebook will test that hypothesis.

In [1]:
# ============================================================
# 1. Build constraint-error mechanism groups
# ============================================================

import numpy as np
import pandas as pd

# Class mapping established in notebook 14
class_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

CONSTRAINT_ID = class_names.index("constraint_error")

# ------------------------------------------------------------
# Find the training labels already present in the notebook
# ------------------------------------------------------------

label_candidates = [
    "y_train",
    "y_train_final",
    "train_labels",
    "y",
]

y_true = None

for name in label_candidates:
    if name in globals():
        candidate = np.asarray(globals()[name])

        if len(candidate) == len(oof_sem_pred):
            y_true = candidate
            print("Using labels:", name, candidate.shape)
            break

if y_true is None:
    raise ValueError(
        "Could not automatically find the 1489 training labels. "
        "Print candidate label arrays and assign y_true manually."
    )

# ------------------------------------------------------------
# Basic correctness
# ------------------------------------------------------------

sem_correct = oof_sem_pred == y_true
trans_correct = oof_trans_pred == y_true

true_constraint = y_true == CONSTRAINT_ID

# Four expert-outcome groups restricted to true constraints
constraint_outcome = np.full(len(y_true), "not_constraint", dtype=object)

constraint_outcome[
    true_constraint & sem_correct & trans_correct
] = "both_correct"

constraint_outcome[
    true_constraint & (~sem_correct) & (~trans_correct)
] = "both_wrong"

constraint_outcome[
    true_constraint & (~sem_correct) & trans_correct
] = "rescue"

constraint_outcome[
    true_constraint & sem_correct & (~trans_correct)
] = "break"

print("Constraint examples:", true_constraint.sum())

print(
    pd.Series(constraint_outcome[true_constraint])
      .value_counts()
)

ValueError: Could not automatically find the 1489 training labels. Print candidate label arrays and assign y_true manually.

In [2]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

CLASS_NAMES = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

CONSTRAINT_ID = CLASS_NAMES.index(
    "constraint_error"
)

In [3]:
# ============================================================
# 2. Load exported trajectory dataset
# ============================================================

train_events = pd.read_csv(
    "../data/processed/trajectory_events_train.csv"
)

test_events = pd.read_csv(
    "../data/processed/trajectory_events_test.csv"
)

targets = pd.read_csv(
    "../data/processed/trajectory_targets.csv"
)

train_targets = (
    targets[
        targets["split"] == "train"
    ]
    .reset_index(drop=True)
)

test_targets = (
    targets[
        targets["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("Train events:", train_events.shape)
print("Test events:", test_events.shape)

print("Train targets:", train_targets.shape)
print("Test targets:", test_targets.shape)

assert len(train_targets) == 1489
assert len(test_targets) == 287

Train events: (3792, 40)
Test events: (799, 40)
Train targets: (1489, 14)
Test targets: (287, 14)


In [4]:
# ============================================================
# 3. Canonical labels and groups
# ============================================================

y_train = (
    train_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

y_test = (
    test_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)


group_col = (
    "canonical_group"
    if "canonical_group"
    in train_targets.columns
    else "group_id"
)

groups_train = (
    train_targets[
        group_col
    ]
    .astype(str)
    .to_numpy()
)

groups_test = (
    test_targets[
        group_col
    ]
    .astype(str)
    .to_numpy()
)


print("Labels:", np.unique(y_train))
print("Train groups:", len(np.unique(groups_train)))
print("Test groups:", len(np.unique(groups_test)))

print("\nTrain distribution:")
print(
    train_targets[
        "failure_family"
    ].value_counts()
)

Labels: [0 1 2 3 4]
Train groups: 335
Test groups: 84

Train distribution:
failure_family
workflow_error           660
constraint_error         317
grounding_state_error    244
tool_use_error           237
reasoning_value_error     31
Name: count, dtype: int64


In [6]:
# ============================================================
# 4. Historical event indices
# ============================================================

TRAJECTORY_KEY = [
    "dataset",
    "group_id",
]


def build_history_indices(
    events_df,
    targets_df,
):

    trajectory_events = {}

    for key, group in events_df.groupby(
        TRAJECTORY_KEY,
        sort=False,
    ):

        trajectory_events[key] = (
            group
            .sort_values(
                "message_index"
            )
        )

    histories = []

    for _, row in targets_df.iterrows():

        key = (
            row["dataset"],
            row["group_id"],
        )

        target_index = int(
            row["message_index"]
        )

        events = trajectory_events.get(
            key
        )

        if events is None:
            histories.append([])
            continue

        history = (
            events.loc[
                events["message_index"]
                < target_index
            ]
            .index
            .tolist()
        )

        histories.append(
            history
        )

    return histories


train_history_indices = (
    build_history_indices(
        train_events,
        train_targets,
    )
)

test_history_indices = (
    build_history_indices(
        test_events,
        test_targets,
    )
)

In [7]:
# ============================================================
# 5. Verify history reconstruction
# ============================================================

train_history_count = np.array([
    len(x)
    for x in train_history_indices
])

test_history_count = np.array([
    len(x)
    for x in test_history_indices
])


np.testing.assert_array_equal(
    train_history_count,
    train_targets[
        "history_event_count"
    ].to_numpy(),
)

np.testing.assert_array_equal(
    test_history_count,
    test_targets[
        "history_event_count"
    ].to_numpy(),
)

print(
    "✓ Historical-event reconstruction exact"
)

print(
    "Train zero-history:",
    (train_history_count == 0).sum(),
)

print(
    "Test zero-history:",
    (test_history_count == 0).sum(),
)

✓ Historical-event reconstruction exact
Train zero-history: 56
Test zero-history: 18


In [8]:
# ============================================================
# 6. Prepare target and historical event text
# ============================================================

if "content" in train_targets.columns:

    target_text_col = "content"

elif "current_text" in train_targets.columns:

    target_text_col = "current_text"

else:
    raise ValueError(
        "No target text column found."
    )


def make_event_text(row):

    role = str(
        row["event_role"]
    ).strip()

    content = (
        ""
        if pd.isna(
            row["content"]
        )
        else str(
            row["content"]
        ).strip()
    )

    prefix = f"[{role}]"

    if content.upper().startswith(
        prefix.upper()
    ):
        return content

    return (
        prefix
        + "\n"
        + content
    )


train_events[
    "event_text"
] = train_events.apply(
    make_event_text,
    axis=1,
)

test_events[
    "event_text"
] = test_events.apply(
    make_event_text,
    axis=1,
)


print(
    "Target text column:",
    target_text_col,
)

Target text column: content


In [12]:
# ============================================================
# 7. Semantic embeddings
# ============================================================

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


train_current_embeddings = (
    encoder.encode(
        train_targets[
            target_text_col
        ]
        .fillna("")
        .astype(str)
        .tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

test_current_embeddings = (
    encoder.encode(
        test_targets[
            target_text_col
        ]
        .fillna("")
        .astype(str)
        .tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)


train_event_embeddings = (
    encoder.encode(
        train_events[
            "event_text"
        ].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

test_event_embeddings = (
    encoder.encode(
        test_events[
            "event_text"
        ].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)


print(
    "Current:",
    train_current_embeddings.shape
)

print(
    "Events:",
    train_event_embeddings.shape
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Current: (1489, 384)
Events: (3792, 384)


In [13]:
# ============================================================
# 8. Last-three history embeddings
# ============================================================

def get_last_k_embeddings(
    history_indices,
    event_embeddings,
    k=3,
):

    n = len(
        history_indices
    )

    d = event_embeddings.shape[1]

    output = np.zeros(
        (n, k, d),
        dtype=np.float32,
    )

    for i, indices in enumerate(
        history_indices
    ):

        if not indices:
            continue

        recent = indices[-k:]

        output[
            i,
            -len(recent):,
            :
        ] = event_embeddings[
            recent
        ]

    return output


H_train_3 = get_last_k_embeddings(
    train_history_indices,
    train_event_embeddings,
    k=3,
)

H_test_3 = get_last_k_embeddings(
    test_history_indices,
    test_event_embeddings,
    k=3,
)

print(
    H_train_3.shape,
    H_test_3.shape,
)

(1489, 3, 384) (287, 3, 384)


In [14]:
# ============================================================
# 9. Compact transition features
# ============================================================

def cosine_rows(a, b):

    num = np.sum(
        a * b,
        axis=1,
    )

    den = (
        np.linalg.norm(
            a,
            axis=1,
        )
        *
        np.linalg.norm(
            b,
            axis=1,
        )
        + 1e-8
    )

    return num / den


def build_transition_features(
    current,
    history,
):

    cols = []
    names = []

    k = history.shape[1]

    for pos in range(k):

        h = history[:, pos, :]

        present = (
            np.linalg.norm(
                h,
                axis=1,
            ) > 1e-8
        ).astype(float)

        cosine = (
            cosine_rows(
                current,
                h,
            )
            * present
        )

        l1 = (
            np.mean(
                np.abs(
                    current - h
                ),
                axis=1,
            )
            * present
        )

        l2 = (
            np.linalg.norm(
                current - h,
                axis=1,
            )
            * present
        )

        relative_pos = (
            k - pos
        )

        cols.extend([
            cosine,
            l1,
            l2,
            present,
        ])

        names.extend([
            f"current_tminus{relative_pos}_cosine",
            f"current_tminus{relative_pos}_l1",
            f"current_tminus{relative_pos}_l2",
            f"tminus{relative_pos}_present",
        ])

    for pos in range(
        1,
        k,
    ):

        older = history[
            :,
            pos - 1,
            :
        ]

        newer = history[
            :,
            pos,
            :
        ]

        pair_present = (
            (
                np.linalg.norm(
                    older,
                    axis=1,
                ) > 1e-8
            )
            &
            (
                np.linalg.norm(
                    newer,
                    axis=1,
                ) > 1e-8
            )
        ).astype(float)

        cosine = (
            cosine_rows(
                older,
                newer,
            )
            * pair_present
        )

        l2 = (
            np.linalg.norm(
                newer - older,
                axis=1,
            )
            * pair_present
        )

        cols.extend([
            cosine,
            l2,
        ])

        names.extend([
            f"history_transition_{pos}_cosine",
            f"history_transition_{pos}_l2",
        ])

    return (
        np.column_stack(
            cols
        ).astype(np.float32),
        names,
    )


T_train, transition_names = (
    build_transition_features(
        train_current_embeddings,
        H_train_3,
    )
)

T_test, _ = (
    build_transition_features(
        test_current_embeddings,
        H_test_3,
    )
)

print(
    "T_train:",
    T_train.shape
)

print(
    transition_names
)

T_train: (1489, 16)
['current_tminus3_cosine', 'current_tminus3_l1', 'current_tminus3_l2', 'tminus3_present', 'current_tminus2_cosine', 'current_tminus2_l1', 'current_tminus2_l2', 'tminus2_present', 'current_tminus1_cosine', 'current_tminus1_l1', 'current_tminus1_l2', 'tminus1_present', 'history_transition_1_cosine', 'history_transition_1_l2', 'history_transition_2_cosine', 'history_transition_2_l2']


In [15]:
# ============================================================
# 10. Expert feature spaces
# ============================================================

X_sem_train = np.asarray(
    train_current_embeddings,
    dtype=np.float32,
)

X_trans_train = np.hstack([
    X_sem_train,
    T_train,
])

print(
    X_sem_train.shape
)

print(
    X_trans_train.shape
)

(1489, 384)
(1489, 400)


In [16]:
# ============================================================
# 11. Reproduce group-safe OOF expert predictions
# ============================================================

N_CLASSES = 5

oof_sem_prob = np.zeros(
    (
        len(y_train),
        N_CLASSES,
    ),
    dtype=np.float32,
)

oof_trans_prob = np.zeros(
    (
        len(y_train),
        N_CLASSES,
    ),
    dtype=np.float32,
)


cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


for fold, (
    tr_idx,
    va_idx,
) in enumerate(
    cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    sem_model = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=0.03,
            max_iter=5000,
            random_state=42,
        ),
    )

    trans_model = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=0.01,
            max_iter=5000,
            random_state=42,
        ),
    )

    sem_model.fit(
        X_sem_train[
            tr_idx
        ],
        y_train[
            tr_idx
        ],
    )

    trans_model.fit(
        X_trans_train[
            tr_idx
        ],
        y_train[
            tr_idx
        ],
    )


    oof_sem_prob[
        va_idx
    ] = sem_model.predict_proba(
        X_sem_train[
            va_idx
        ]
    )

    oof_trans_prob[
        va_idx
    ] = trans_model.predict_proba(
        X_trans_train[
            va_idx
        ]
    )

    print(
        f"Fold {fold} complete"
    )


oof_sem_pred = (
    oof_sem_prob.argmax(
        axis=1
    )
)

oof_trans_pred = (
    oof_trans_prob.argmax(
        axis=1
    )
)

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [17]:
# ============================================================
# 12. Reproduction check
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)


def metrics(pred):

    return {
        "accuracy":
            accuracy_score(
                y_train,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_train,
                pred,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                y_train,
                pred,
                average="weighted",
            ),
    }


print(
    "Semantic:",
    metrics(
        oof_sem_pred
    )
)

print(
    "Transition:",
    metrics(
        oof_trans_pred
    )
)


sem_correct = (
    oof_sem_pred
    == y_train
)

trans_correct = (
    oof_trans_pred
    == y_train
)


print(
    "\nRescues:",
    (
        (~sem_correct)
        &
        trans_correct
    ).sum()
)

print(
    "Breaks:",
    (
        sem_correct
        &
        (~trans_correct)
    ).sum()
)

Semantic: {'accuracy': 0.5251846877098724, 'balanced_accuracy': 0.49075604688714486, 'macro_f1': 0.49741135402084663, 'weighted_f1': 0.5230711265222063}
Transition: {'accuracy': 0.5352585627938213, 'balanced_accuracy': 0.4838982558191824, 'macro_f1': 0.4980829248130962, 'weighted_f1': 0.5322960023974649}

Rescues: 69
Breaks: 54


In [18]:
# ============================================================
# 13. Constraint-error mechanism groups
# ============================================================

true_constraint = (
    y_train
    == CONSTRAINT_ID
)

sem_correct = (
    oof_sem_pred
    == y_train
)

trans_correct = (
    oof_trans_pred
    == y_train
)


constraint_outcome = np.full(
    len(y_train),
    "not_constraint",
    dtype=object,
)


constraint_outcome[
    true_constraint
    & sem_correct
    & trans_correct
] = "both_correct"


constraint_outcome[
    true_constraint
    & (~sem_correct)
    & (~trans_correct)
] = "both_wrong"


constraint_outcome[
    true_constraint
    & (~sem_correct)
    & trans_correct
] = "rescue"


constraint_outcome[
    true_constraint
    & sem_correct
    & (~trans_correct)
] = "break"


print(
    "Constraint examples:",
    true_constraint.sum()
)

print(
    pd.Series(
        constraint_outcome[
            true_constraint
        ]
    ).value_counts()
)

Constraint examples: 317
both_wrong      170
both_correct    114
rescue           26
break             7
Name: count, dtype: int64


In [19]:
# ============================================================
# 14. Build constraint mechanism dataframe
# ============================================================

constraint_idx = np.where(
    true_constraint
)[0]

constraint_df = pd.DataFrame({
    "row_idx": constraint_idx,

    "outcome": constraint_outcome[
        constraint_idx
    ],

    "semantic_pred":
        oof_sem_pred[
            constraint_idx
        ],

    "transition_pred":
        oof_trans_pred[
            constraint_idx
        ],

    "semantic_pred_name": [
        CLASS_NAMES[int(i)]
        for i in oof_sem_pred[
            constraint_idx
        ]
    ],

    "transition_pred_name": [
        CLASS_NAMES[int(i)]
        for i in oof_trans_pred[
            constraint_idx
        ]
    ],

    "semantic_constraint_prob":
        oof_sem_prob[
            constraint_idx,
            CONSTRAINT_ID,
        ],

    "transition_constraint_prob":
        oof_trans_prob[
            constraint_idx,
            CONSTRAINT_ID,
        ],
})

constraint_df[
    "constraint_prob_gain"
] = (
    constraint_df[
        "transition_constraint_prob"
    ]
    -
    constraint_df[
        "semantic_constraint_prob"
    ]
)

constraint_df[
    "history_event_count"
] = (
    train_targets.loc[
        constraint_idx,
        "history_event_count",
    ]
    .to_numpy()
)

print(
    constraint_df[
        "outcome"
    ].value_counts()
)

display(
    constraint_df.head()
)

outcome
both_wrong      170
both_correct    114
rescue           26
break             7
Name: count, dtype: int64


,row_idx,outcome,semantic_pred,transition_pred,semantic_pred_name,transition_pred_name,semantic_constraint_prob,transition_constraint_prob,constraint_prob_gain,history_event_count
0,0,both_wrong,3,3,grounding_state_error,grounding_state_error,0.002098,0.016013,0.013915,2
1,3,both_correct,1,1,constraint_error,constraint_error,0.883326,0.832058,-0.051268,3
2,4,both_correct,1,1,constraint_error,constraint_error,0.730519,0.629578,-0.100941,3
3,5,rescue,0,1,workflow_error,constraint_error,0.356385,0.454329,0.097944,0
4,6,both_correct,1,1,constraint_error,constraint_error,0.686879,0.599621,-0.087258,0


In [20]:
# ============================================================
# 15. Attach compact trajectory features
# ============================================================

transition_feature_df = pd.DataFrame(
    T_train,
    columns=transition_names,
)

for feature in transition_names:

    constraint_df[
        feature
    ] = (
        transition_feature_df.loc[
            constraint_idx,
            feature,
        ]
        .to_numpy()
    )

print(
    "Constraint dataframe:",
    constraint_df.shape
)

Constraint dataframe: (317, 26)


In [21]:
# ============================================================
# 16. Compare the four constraint outcomes
# ============================================================

core_features = [
    "semantic_constraint_prob",
    "transition_constraint_prob",
    "constraint_prob_gain",
    "history_event_count",

    "current_tminus1_cosine",
    "current_tminus2_cosine",
    "current_tminus3_cosine",

    "current_tminus1_l2",
    "current_tminus2_l2",
    "current_tminus3_l2",

    "history_transition_1_cosine",
    "history_transition_2_cosine",

    "tminus1_present",
    "tminus2_present",
    "tminus3_present",
]

constraint_group_summary = (
    constraint_df
    .groupby(
        "outcome"
    )[core_features]
    .agg([
        "count",
        "mean",
        "median",
        "std",
    ])
)

display(
    constraint_group_summary.round(4)
)

semantic_constraint_prob                          \
                                count    mean  median     std   
outcome                                                         
both_correct                      114  0.6531  0.6307  0.1719   
both_wrong                        170  0.1450  0.1141  0.1205   
break                               7  0.4879  0.4991  0.0621   
rescue                             26  0.3432  0.3508  0.0846   

             transition_constraint_prob                          \
                                  count    mean  median     std   
outcome                                                           
both_correct                        114  0.6077  0.5851  0.1460   
both_wrong                          170  0.1697  0.1464  0.1111   
break                                 7  0.3256  0.3084  0.0988   
rescue                               26  0.4312  0.4369  0.0599   

             constraint_prob_gain          ... tminus1_present          \
                            count    mean  ...          median     std   
outcome                                    ...                           
both_correct                  114 -0.0454  ...             1.0  0.1848   
both_wrong                    170  0.0247  ...             1.0  0.2124   
break                           7 -0.1623  ...             1.0  0.0000   
rescue                         26  0.0880  ...             1.0  0.1961   

             tminus2_present                        tminus3_present          \
                       count    mean median     std           count    mean   
outcome                                                                       
both_correct             114  0.9649    1.0  0.1848             114  0.9474   
both_wrong               170  0.9118    1.0  0.2845             170  0.8765   
break                      7  0.7143    1.0  0.4880               7  0.7143   
rescue                    26  0.9615    1.0  0.1961              26  0.9231   

                             
             median     std  
outcome                      
both_correct    1.0  0.2243  
both_wrong      1.0  0.3300  
break           1.0  0.4880  
rescue          1.0  0.2717  

[4 rows x 60 columns]

In [22]:
# ============================================================
# 17. Rescue vs both-wrong effect sizes
# ============================================================

def cohens_d(
    group_a,
    group_b,
):
    a = np.asarray(
        group_a,
        dtype=float,
    )

    b = np.asarray(
        group_b,
        dtype=float,
    )

    a = a[
        ~np.isnan(a)
    ]

    b = b[
        ~np.isnan(b)
    ]

    if (
        len(a) < 2
        or len(b) < 2
    ):
        return np.nan

    pooled_var = (
        (
            (len(a) - 1)
            * np.var(a, ddof=1)
        )
        +
        (
            (len(b) - 1)
            * np.var(b, ddof=1)
        )
    ) / (
        len(a)
        + len(b)
        - 2
    )

    pooled_sd = np.sqrt(
        pooled_var
    )

    if pooled_sd == 0:
        return 0.0

    return (
        np.mean(a)
        - np.mean(b)
    ) / pooled_sd


comparison_rows = []

for feature in core_features:

    rescue_values = (
        constraint_df.loc[
            constraint_df[
                "outcome"
            ] == "rescue",
            feature,
        ]
    )

    wrong_values = (
        constraint_df.loc[
            constraint_df[
                "outcome"
            ] == "both_wrong",
            feature,
        ]
    )

    comparison_rows.append({
        "feature":
            feature,

        "rescue_mean":
            rescue_values.mean(),

        "both_wrong_mean":
            wrong_values.mean(),

        "difference":
            (
                rescue_values.mean()
                -
                wrong_values.mean()
            ),

        "cohens_d":
            cohens_d(
                rescue_values,
                wrong_values,
            ),
    })


rescue_vs_wrong = (
    pd.DataFrame(
        comparison_rows
    )
)

rescue_vs_wrong[
    "abs_d"
] = (
    rescue_vs_wrong[
        "cohens_d"
    ].abs()
)

rescue_vs_wrong = (
    rescue_vs_wrong
    .sort_values(
        "abs_d",
        ascending=False,
    )
    .drop(
        columns="abs_d"
    )
)

display(
    rescue_vs_wrong.round(4)
)

,feature,rescue_mean,both_wrong_mean,difference,cohens_d
1,transition_constraint_prob,0.4312,0.1697,0.2615,2.4701
0,semantic_constraint_prob,0.3432,0.1450,0.1982,1.7017
2,constraint_prob_gain,0.0880,0.0247,0.0633,1.2440
9,current_tminus3_l2,1.0639,0.8745,0.1894,0.4478
6,current_tminus3_cosine,0.2983,0.4010,-0.1027,-0.3698
3,history_event_count,9.2308,14.5353,-5.3045,-0.3443
8,current_tminus2_l2,0.9621,0.8881,0.0739,0.1991
13,tminus2_present,0.9615,0.9118,0.0498,0.1812
14,tminus3_present,0.9231,0.8765,0.0466,0.1442
7,current_tminus1_l2,0.9242,0.8904,0.0338,0.1036


In [23]:
# ============================================================
# 18. Which semantic confusions are rescued?
# ============================================================

constraint_rescues = (
    constraint_df[
        constraint_df[
            "outcome"
        ] == "rescue"
    ]
    .copy()
)

constraint_breaks = (
    constraint_df[
        constraint_df[
            "outcome"
        ] == "break"
    ]
    .copy()
)

constraint_both_wrong = (
    constraint_df[
        constraint_df[
            "outcome"
        ] == "both_wrong"
    ]
    .copy()
)


print(
    "RESCUE SOURCE PREDICTIONS"
)

display(
    constraint_rescues[
        "semantic_pred_name"
    ]
    .value_counts()
    .rename_axis(
        "semantic_pred"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nBOTH-WRONG SEMANTIC PREDICTIONS"
)

display(
    constraint_both_wrong[
        "semantic_pred_name"
    ]
    .value_counts()
    .rename_axis(
        "semantic_pred"
    )
    .reset_index(
        name="count"
    )
)

RESCUE SOURCE PREDICTIONS


,semantic_pred,count
0,workflow_error,15
1,tool_use_error,7
2,grounding_state_error,4



BOTH-WRONG SEMANTIC PREDICTIONS


,semantic_pred,count
0,workflow_error,99
1,grounding_state_error,50
2,tool_use_error,18
3,reasoning_value_error,3


In [24]:
# ============================================================
# 19. How trajectory changes constraint probability
# ============================================================

probability_summary = (
    constraint_df
    .groupby(
        "outcome"
    )
    .agg(
        count=(
            "outcome",
            "size",
        ),

        semantic_constraint_prob=(
            "semantic_constraint_prob",
            "mean",
        ),

        transition_constraint_prob=(
            "transition_constraint_prob",
            "mean",
        ),

        mean_constraint_gain=(
            "constraint_prob_gain",
            "mean",
        ),

        median_constraint_gain=(
            "constraint_prob_gain",
            "median",
        ),
    )
)

display(
    probability_summary
    .round(4)
)

,count,semantic_constraint_prob,transition_constraint_prob,mean_constraint_gain,median_constraint_gain
outcome,,,,,
both_correct,114,0.6531,0.6077,-0.0454,-0.0565
both_wrong,170,0.1450,0.1697,0.0247,0.0241
break,7,0.4879,0.3256,-0.1623,-0.1858
rescue,26,0.3432,0.4312,0.0880,0.0830


In [26]:
# ============================================================
# 20. Constraint rank before and after trajectory
# ============================================================

def class_rank(
    probabilities,
    class_id,
):

    order = np.argsort(
        -probabilities,
        axis=1,
    )

    ranks = np.empty(
        len(probabilities),
        dtype=int,
    )

    for i in range(
        len(probabilities)
    ):
        ranks[i] = (
            np.where(
                order[i]
                == class_id
            )[0][0]
            + 1
        )

    return ranks


constraint_semantic_rank_all = (
    class_rank(
        oof_sem_prob,
        CONSTRAINT_ID,
    )
)

constraint_transition_rank_all = (
    class_rank(
        oof_trans_prob,
        CONSTRAINT_ID,
    )
)


constraint_df[
    "semantic_constraint_rank"
] = (
    constraint_semantic_rank_all[
        constraint_idx
    ]
)

constraint_df[
    "transition_constraint_rank"
] = (
    constraint_transition_rank_all[
        constraint_idx
    ]
)

constraint_df[
    "rank_improvement"
] = (
    constraint_df[
        "semantic_constraint_rank"
    ]
    -
    constraint_df[
        "transition_constraint_rank"
    ]
)


rank_summary = (
    constraint_df
    .groupby(
        "outcome"
    )
    .agg(
        count=(
            "outcome",
            "size",
        ),

        mean_semantic_rank=(
            "semantic_constraint_rank",
            "mean",
        ),

        mean_transition_rank=(
            "transition_constraint_rank",
            "mean",
        ),

        mean_rank_improvement=(
            "rank_improvement",
            "mean",
        ),

        pct_semantic_rank2=(
            "semantic_constraint_rank",
            lambda x:
                (x == 2).mean(),
        ),
    )
)

display(
    rank_summary.round(4)
)

,count,mean_semantic_rank,mean_transition_rank,mean_rank_improvement,pct_semantic_rank2
outcome,,,,,
both_correct,114,1.0000,1.0000,0.0000,0.0000
both_wrong,170,2.7824,2.7882,-0.0059,0.4353
break,7,1.0000,2.1429,-1.1429,0.0000
rescue,26,2.1154,1.0000,1.1154,0.8846


In [27]:
# ============================================================
# 21. History depth distribution by constraint outcome
# ============================================================

history_depth_summary = (
    constraint_df
    .groupby(
        "outcome"
    )[
        "history_event_count"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
        ]
    )
)

display(
    history_depth_summary
    .round(4)
)


constraint_df[
    "history_depth_bin"
] = pd.cut(
    constraint_df[
        "history_event_count"
    ],
    bins=[
        -1,
        0,
        3,
        10,
        20,
        np.inf,
    ],
    labels=[
        "0",
        "1_3",
        "4_10",
        "11_20",
        "21_plus",
    ],
)


history_outcome_table = pd.crosstab(
    constraint_df[
        "history_depth_bin"
    ],
    constraint_df[
        "outcome"
    ],
    normalize="index",
)

display(
    history_outcome_table
    .round(4)
)

,count,mean,median,std,min,max
outcome,,,,,,
both_correct,114,8.3421,8.0,5.3923,0,41
both_wrong,170,14.5353,10.0,16.4148,0,93
break,7,8.5714,7.0,7.7644,1,22
rescue,26,9.2308,9.0,4.4838,0,18


outcome,both_correct,both_wrong,break,rescue
history_depth_bin,,,,
0,0.3077,0.6154,0.0000,0.0769
1_3,0.2593,0.6296,0.0741,0.0370
4_10,0.4812,0.4188,0.0188,0.0812
11_20,0.3086,0.5432,0.0123,0.1358
21_plus,0.0278,0.9444,0.0278,0.0000


What textual/structural patterns in recent history distinguish rescued constraint errors from unresolved constraint errors?

In [28]:
# ============================================================
# 22. Build readable rescue examples
# ============================================================

def get_recent_history_rows(
    row_idx,
    k=5,
):
    history_indices = (
        train_history_indices[
            row_idx
        ]
    )

    recent = history_indices[-k:]

    if not recent:
        return pd.DataFrame(
            columns=[
                "message_index",
                "event_role",
                "content",
            ]
        )

    return (
        train_events
        .loc[
            recent,
            [
                "message_index",
                "event_role",
                "content",
            ],
        ]
        .copy()
    )


rescue_indices = (
    constraint_df.loc[
        constraint_df["outcome"]
        == "rescue",
        "row_idx",
    ]
    .astype(int)
    .tolist()
)

both_wrong_indices = (
    constraint_df.loc[
        constraint_df["outcome"]
        == "both_wrong",
        "row_idx",
    ]
    .astype(int)
    .tolist()
)

print("Rescues:", len(rescue_indices))
print("Both wrong:", len(both_wrong_indices))

Rescues: 26
Both wrong: 170


In [29]:
# ============================================================
# 23. Export readable rescue trajectories
# ============================================================

rescue_records = []

for row_idx in rescue_indices:

    target = train_targets.iloc[
        row_idx
    ]

    history = get_recent_history_rows(
        row_idx,
        k=5,
    )

    history_text = "\n\n".join([
        (
            f"[{r.event_role}] "
            f"{str(r.content)}"
        )
        for _, r in history.iterrows()
    ])

    rescue_records.append({
        "row_idx":
            row_idx,

        "dataset":
            target["dataset"],

        "group_id":
            target["group_id"],

        "message_index":
            target["message_index"],

        "semantic_prediction":
            CLASS_NAMES[
                int(
                    oof_sem_pred[
                        row_idx
                    ]
                )
            ],

        "semantic_constraint_prob":
            oof_sem_prob[
                row_idx,
                CONSTRAINT_ID,
            ],

        "transition_constraint_prob":
            oof_trans_prob[
                row_idx,
                CONSTRAINT_ID,
            ],

        "constraint_prob_gain":
            (
                oof_trans_prob[
                    row_idx,
                    CONSTRAINT_ID,
                ]
                -
                oof_sem_prob[
                    row_idx,
                    CONSTRAINT_ID,
                ]
            ),

        "history_event_count":
            train_targets.iloc[
                row_idx
            ][
                "history_event_count"
            ],

        "recent_history":
            history_text,

        "target_text":
            train_targets.iloc[
                row_idx
            ][
                target_text_col
            ],
    })


rescue_examples_df = pd.DataFrame(
    rescue_records
)

display(
    rescue_examples_df[
        [
            "row_idx",
            "semantic_prediction",
            "semantic_constraint_prob",
            "transition_constraint_prob",
            "constraint_prob_gain",
            "history_event_count",
        ]
    ]
    .sort_values(
        "constraint_prob_gain",
        ascending=False,
    )
    .round(4)
)

,row_idx,semantic_prediction,semantic_constraint_prob,transition_constraint_prob,constraint_prob_gain,history_event_count
24,1112,tool_use_error,0.2098,0.4157,0.2059,11
10,699,grounding_state_error,0.1762,0.3670,0.1909,9
23,1039,workflow_error,0.3060,0.4459,0.1399,10
9,684,tool_use_error,0.1733,0.3067,0.1334,8
22,1023,grounding_state_error,0.2928,0.4240,0.1311,5
5,475,tool_use_error,0.3806,0.5038,0.1231,12
8,682,tool_use_error,0.2550,0.3757,0.1207,6
15,899,workflow_error,0.3485,0.4689,0.1205,13
18,973,workflow_error,0.3117,0.4247,0.1131,6
11,831,tool_use_error,0.3132,0.4213,0.1080,13


In [30]:
# ============================================================
# 24. Print strongest trajectory rescues
# ============================================================

top_rescues = (
    rescue_examples_df
    .sort_values(
        "constraint_prob_gain",
        ascending=False,
    )
    .head(10)
)

for _, row in top_rescues.iterrows():

    print("\n" + "=" * 100)
    print(
        "ROW:",
        row["row_idx"],
        "| semantic:",
        row[
            "semantic_prediction"
        ],
    )

    print(
        "P(constraint):",
        round(
            row[
                "semantic_constraint_prob"
            ],
            3,
        ),
        "->",
        round(
            row[
                "transition_constraint_prob"
            ],
            3,
        ),
    )

    print(
        "\nRECENT HISTORY\n"
    )

    print(
        row[
            "recent_history"
        ]
    )

    print(
        "\nTARGET\n"
    )

    print(
        row[
            "target_text"
        ]
    )


ROW: 1112 | semantic: tool_use_error
P(constraint): 0.21 -> 0.416

RECENT HISTORY

[ASSISTANT] [ASSISTANT]
The estimated distance between San Francisco and Rivermist is 980.0 kilometers. This information can help you plan your fuel consumption and travel time accordingly.

[TOOL_CALL] [TOOL_CALL]
<tool_call>
{"name": "posting_get_login_status", "arguments": {}}
</tool_call>
posting_get_login_status({})

[TOOL_CALL] [TOOL_CALL]
<tool_call>
{"name": "post_tweet", "arguments": {"content": "Excited for my trip from San Francisco to Rivermist!", "mentions": ["@TravelBuddy"], "tags": ["#JourneyAhead"]}}
</tool_call>
post_tweet({"content": "Excited for my trip from San Francisco to Rivermist!", "mentions": ["@TravelBuddy"], "tags": ["#JourneyAhead"]})

[ASSISTANT] [ASSISTANT]
I have posted the following tweet:  
"Excited for my trip from San Francisco to Rivermist! #JourneyAhead @TravelBuddy"  

The tweet has been successfully shared with your followers.

[TOOL_CALL] [TOOL_CALL]
<tool_call>


In [31]:
# ============================================================
# 25. Match rescues with unresolved cases by semantic plausibility
# ============================================================

wrong_pool = (
    constraint_df[
        constraint_df[
            "outcome"
        ] == "both_wrong"
    ]
    .copy()
)

matched_pairs = []

for _, rescue_row in (
    constraint_df[
        constraint_df[
            "outcome"
        ] == "rescue"
    ]
    .iterrows()
):

    candidate_pool = (
        wrong_pool.copy()
    )

    candidate_pool[
        "prob_distance"
    ] = np.abs(
        candidate_pool[
            "semantic_constraint_prob"
        ]
        -
        rescue_row[
            "semantic_constraint_prob"
        ]
    )

    match = (
        candidate_pool
        .sort_values(
            "prob_distance"
        )
        .iloc[0]
    )

    matched_pairs.append({
        "rescue_idx":
            int(
                rescue_row[
                    "row_idx"
                ]
            ),

        "wrong_idx":
            int(
                match[
                    "row_idx"
                ]
            ),

        "rescue_sem_prob":
            rescue_row[
                "semantic_constraint_prob"
            ],

        "wrong_sem_prob":
            match[
                "semantic_constraint_prob"
            ],

        "rescue_gain":
            rescue_row[
                "constraint_prob_gain"
            ],

        "wrong_gain":
            match[
                "constraint_prob_gain"
            ],
    })


matched_pairs_df = pd.DataFrame(
    matched_pairs
)

display(
    matched_pairs_df
    .round(4)
)

,rescue_idx,wrong_idx,rescue_sem_prob,wrong_sem_prob,rescue_gain,wrong_gain
0,5,139,0.3564,0.3563,0.0979,0.0072
1,109,821,0.2928,0.2923,0.0458,-0.0069
2,141,376,0.3339,0.3365,0.0633,-0.0862
3,414,724,0.2395,0.2350,0.1031,-0.0209
4,472,1436,0.3257,0.3296,0.0123,-0.0391
5,475,855,0.3806,0.3748,0.1231,-0.0765
6,570,704,0.4294,0.4186,0.0072,0.0026
7,592,855,0.3718,0.3748,0.0812,-0.0765
8,682,7,0.2550,0.2528,0.1207,-0.0043
9,684,1281,0.1733,0.1727,0.1334,0.0057


In [32]:
# ============================================================
# 26. Inspect matched rescue vs unresolved examples
# ============================================================

for _, pair in (
    matched_pairs_df
    .head(10)
    .iterrows()
):

    rescue_idx = int(
        pair["rescue_idx"]
    )

    wrong_idx = int(
        pair["wrong_idx"]
    )

    print("\n" + "#" * 100)
    print("RESCUE")
    print("#" * 100)

    print(
        "\nHistory:"
    )

    print(
        "\n".join(
            get_recent_history_rows(
                rescue_idx,
                k=5,
            )[
                "content"
            ]
            .fillna("")
            .astype(str)
            .tolist()
        )
    )

    print(
        "\nTarget:"
    )

    print(
        train_targets.iloc[
            rescue_idx
        ][target_text_col]
    )


    print("\n" + "-" * 100)
    print("BOTH WRONG")
    print("-" * 100)

    print(
        "\nHistory:"
    )

    print(
        "\n".join(
            get_recent_history_rows(
                wrong_idx,
                k=5,
            )[
                "content"
            ]
            .fillna("")
            .astype(str)
            .tolist()
        )
    )

    print(
        "\nTarget:"
    )

    print(
        train_targets.iloc[
            wrong_idx
        ][target_text_col]
    )


####################################################################################################
RESCUE
####################################################################################################

History:


Target:
[ASSISTANT]
<answer>I'm a Jayhawk</answer>

----------------------------------------------------------------------------------------------------
BOTH WRONG
----------------------------------------------------------------------------------------------------

History:
[TOOL_CALL]

get_product_details({"product_id": "6086499569"})
[ASSISTANT]
I apologize for the error. It seems there was an issue retrieving the skateboard product details. Let me verify the correct product ID for skateboards. 

Let me check the list of all product types to find the correct one for skateboards. 

Would you like me to proceed with that?
[TOOL_CALL]

list_all_product_types({})
[ASSISTANT]
I see that the product ID for skateboards is **1968349452**. Let me retrieve the details for thi

In [33]:
# ============================================================
# 27. Research control:
#     Isolate the causal contribution of trajectory features
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

N_CLASSES = len(CLASS_NAMES)

# Predictions from identical transition models under:
#
# 1. actual trajectory features
# 2. trajectory features neutralized to the TRAIN-FOLD mean
#
# After StandardScaler, the fold mean corresponds to approximately
# zero standardized trajectory signal.

oof_transition_actual_prob = np.zeros(
    (len(y_train), N_CLASSES),
    dtype=np.float64,
)

oof_transition_neutral_prob = np.zeros(
    (len(y_train), N_CLASSES),
    dtype=np.float64,
)

# Also reproduce a semantic-only model with the SAME C as the
# transition model. This separates regularization effects.
oof_semantic_sameC_prob = np.zeros(
    (len(y_train), N_CLASSES),
    dtype=np.float64,
)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (tr_idx, va_idx) in enumerate(
    cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    # --------------------------------------------------------
    # Transition model
    # --------------------------------------------------------

    X_tr_actual = np.hstack([
        X_sem_train[tr_idx],
        T_train[tr_idx],
    ])

    X_va_actual = np.hstack([
        X_sem_train[va_idx],
        T_train[va_idx],
    ])

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(
        X_tr_actual
    )

    X_va_actual_scaled = scaler.transform(
        X_va_actual
    )

    transition_model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    transition_model.fit(
        X_tr_scaled,
        y_train[tr_idx],
    )

    oof_transition_actual_prob[
        va_idx
    ] = transition_model.predict_proba(
        X_va_actual_scaled
    )


    # --------------------------------------------------------
    # Neutralize trajectory features
    #
    # Use training-fold mean of each trajectory feature.
    # StandardScaler maps these approximately to zero.
    # --------------------------------------------------------

    trajectory_mean = (
        T_train[tr_idx]
        .mean(axis=0)
    )

    T_va_neutral = np.tile(
        trajectory_mean,
        (
            len(va_idx),
            1,
        ),
    )

    X_va_neutral = np.hstack([
        X_sem_train[va_idx],
        T_va_neutral,
    ])

    X_va_neutral_scaled = scaler.transform(
        X_va_neutral
    )

    oof_transition_neutral_prob[
        va_idx
    ] = transition_model.predict_proba(
        X_va_neutral_scaled
    )


    # --------------------------------------------------------
    # Semantic-only control with transition C=0.01
    # --------------------------------------------------------

    sem_scaler = StandardScaler()

    X_sem_tr_scaled = (
        sem_scaler.fit_transform(
            X_sem_train[tr_idx]
        )
    )

    X_sem_va_scaled = (
        sem_scaler.transform(
            X_sem_train[va_idx]
        )
    )

    semantic_sameC_model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    semantic_sameC_model.fit(
        X_sem_tr_scaled,
        y_train[tr_idx],
    )

    oof_semantic_sameC_prob[
        va_idx
    ] = semantic_sameC_model.predict_proba(
        X_sem_va_scaled
    )

    print(
        f"Fold {fold} complete"
    )

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [34]:
# ============================================================
# 28. Controlled predictions
# ============================================================

pred_semantic_original = (
    oof_sem_pred.copy()
)

pred_semantic_sameC = (
    oof_semantic_sameC_prob
    .argmax(axis=1)
)

pred_transition_neutral = (
    oof_transition_neutral_prob
    .argmax(axis=1)
)

pred_transition_actual = (
    oof_transition_actual_prob
    .argmax(axis=1)
)


control_results = pd.DataFrame([
    {
        "model":
            "semantic_original_C003",
        **metrics(
            pred_semantic_original
        ),
    },
    {
        "model":
            "semantic_same_C001",
        **metrics(
            pred_semantic_sameC
        ),
    },
    {
        "model":
            "transition_neutralized",
        **metrics(
            pred_transition_neutral
        ),
    },
    {
        "model":
            "transition_actual",
        **metrics(
            pred_transition_actual
        ),
    },
])

display(
    control_results.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_original_C003,0.5252,0.4908,0.4974,0.5231
1,semantic_same_C001,0.5292,0.4777,0.4908,0.5257
2,transition_neutralized,0.5306,0.4761,0.4939,0.5255
3,transition_actual,0.5353,0.4839,0.4981,0.5323


In [35]:
# ============================================================
# 29. Genuine trajectory contribution for constraint errors
# ============================================================

constraint_mask = (
    y_train == CONSTRAINT_ID
)

neutral_correct = (
    pred_transition_neutral
    == y_train
)

actual_correct = (
    pred_transition_actual
    == y_train
)


# Real trajectory feature causes a correction:
trajectory_rescue = (
    constraint_mask
    &
    (~neutral_correct)
    &
    actual_correct
)

# Real trajectory feature destroys a correct prediction:
trajectory_break = (
    constraint_mask
    &
    neutral_correct
    &
    (~actual_correct)
)

# Unchanged correctness
trajectory_both_correct = (
    constraint_mask
    &
    neutral_correct
    &
    actual_correct
)

trajectory_both_wrong = (
    constraint_mask
    &
    (~neutral_correct)
    &
    (~actual_correct)
)


print(
    "TRUE CONSTRAINT EXAMPLES:",
    constraint_mask.sum(),
)

print(
    "\nGenuine trajectory rescues:",
    trajectory_rescue.sum(),
)

print(
    "Genuine trajectory breaks:",
    trajectory_break.sum(),
)

print(
    "Net trajectory contribution:",
    (
        trajectory_rescue.sum()
        -
        trajectory_break.sum()
    ),
)

print(
    "\nBoth correct:",
    trajectory_both_correct.sum(),
)

print(
    "Both wrong:",
    trajectory_both_wrong.sum(),
)

TRUE CONSTRAINT EXAMPLES: 317

Genuine trajectory rescues: 13
Genuine trajectory breaks: 5
Net trajectory contribution: 8

Both correct: 127
Both wrong: 172


In [36]:
# ============================================================
# 30. Apparent rescue vs genuine trajectory rescue
# ============================================================

original_constraint_rescue = (
    constraint_mask
    &
    (pred_semantic_original != y_train)
    &
    (pred_transition_actual == y_train)
)


print(
    "Original apparent rescues:",
    original_constraint_rescue.sum(),
)

print(
    "Genuine trajectory rescues:",
    trajectory_rescue.sum(),
)

print(
    "Original rescues that are genuine:",
    (
        original_constraint_rescue
        &
        trajectory_rescue
    ).sum(),
)

print(
    "Original rescues NOT attributable "
    "to trajectory features:",
    (
        original_constraint_rescue
        &
        (~trajectory_rescue)
    ).sum(),
)

Original apparent rescues: 26
Genuine trajectory rescues: 13
Original rescues that are genuine: 9
Original rescues NOT attributable to trajectory features: 17


In [37]:
# ============================================================
# 31. Zero-history sanity check
# ============================================================

zero_history = (
    train_targets[
        "history_event_count"
    ].to_numpy()
    == 0
)

print(
    "Zero-history true constraints:",
    (
        zero_history
        &
        constraint_mask
    ).sum(),
)

print(
    "Original apparent rescues "
    "with zero history:",
    (
        zero_history
        &
        original_constraint_rescue
    ).sum(),
)

print(
    "Genuine trajectory rescues "
    "with zero history:",
    (
        zero_history
        &
        trajectory_rescue
    ).sum(),
)

Zero-history true constraints: 13
Original apparent rescues with zero history: 1
Genuine trajectory rescues with zero history: 2


In [39]:
print("T_train shape:", T_train.shape)
print("Number of feature names:", len(transition_names))

for i, name in enumerate(transition_names):
    print(f"{i:2d}: {name}")

T_train shape: (1489, 16)
Number of feature names: 16
 0: current_tminus3_cosine
 1: current_tminus3_l1
 2: current_tminus3_l2
 3: tminus3_present
 4: current_tminus2_cosine
 5: current_tminus2_l1
 6: current_tminus2_l2
 7: tminus2_present
 8: current_tminus1_cosine
 9: current_tminus1_l1
10: current_tminus1_l2
11: tminus1_present
12: history_transition_1_cosine
13: history_transition_1_l2
14: history_transition_2_cosine
15: history_transition_2_l2


In [40]:
# ============================================================
# 32. Diagnose zero-history trajectory rescues
# ============================================================

zero_history_rescue_idx = np.where(
    zero_history
    &
    trajectory_rescue
)[0]

print(
    "Zero-history trajectory rescues:",
    zero_history_rescue_idx.tolist(),
)

zero_history_feature_rows = (
    train_targets
    .iloc[zero_history_rescue_idx]
    .copy()
)

for j, col in enumerate(
    transition_names
):
    zero_history_feature_rows[
        col
    ] = T_train[
        zero_history_rescue_idx,
        j,
    ]

zero_history_feature_rows[
    "neutral_pred"
] = [
    CLASS_NAMES[int(i)]
    for i in pred_transition_neutral[
        zero_history_rescue_idx
    ]
]

zero_history_feature_rows[
    "actual_pred"
] = [
    CLASS_NAMES[int(i)]
    for i in pred_transition_actual[
        zero_history_rescue_idx
    ]
]

zero_history_feature_rows[
    "true_label"
] = [
    CLASS_NAMES[int(i)]
    for i in y_train[
        zero_history_rescue_idx
    ]
]

display_cols = [
    "neutral_pred",
    "actual_pred",
    "true_label",
    "history_event_count",
] + list(transition_names)

display(
    zero_history_feature_rows[
        display_cols
    ].T
)

Zero-history trajectory rescues: [5, 1181]


,5,1181
neutral_pred,workflow_error,grounding_state_error
actual_pred,constraint_error,constraint_error
true_label,constraint_error,constraint_error
history_event_count,0,0
current_tminus3_cosine,0.0,0.0
current_tminus3_l1,0.0,0.0
current_tminus3_l2,0.0,0.0
tminus3_present,0.0,0.0
current_tminus2_cosine,0.0,0.0
current_tminus2_l1,0.0,0.0


In [41]:
# ============================================================
# 33. Do transition features vary when history is absent?
# ============================================================

T_zero = T_train[
    zero_history
]

zero_history_variation = pd.DataFrame({
    "feature":
        transition_names,

    "mean":
        T_zero.mean(axis=0),

    "std":
        T_zero.std(axis=0),

    "min":
        T_zero.min(axis=0),

    "max":
        T_zero.max(axis=0),

    "n_unique": [
        len(
            np.unique(
                T_zero[:, j]
            )
        )
        for j in range(
            T_zero.shape[1]
        )
    ],
})

zero_history_variation[
    "varies_without_history"
] = (
    zero_history_variation[
        "n_unique"
    ] > 1
)

display(
    zero_history_variation
    .sort_values(
        [
            "varies_without_history",
            "std",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

,feature,mean,std,min,max,n_unique,varies_without_history
0,current_tminus3_cosine,0.0,0.0,0.0,0.0,1,False
1,current_tminus3_l1,0.0,0.0,0.0,0.0,1,False
2,current_tminus3_l2,0.0,0.0,0.0,0.0,1,False
3,tminus3_present,0.0,0.0,0.0,0.0,1,False
4,current_tminus2_cosine,0.0,0.0,0.0,0.0,1,False
5,current_tminus2_l1,0.0,0.0,0.0,0.0,1,False
6,current_tminus2_l2,0.0,0.0,0.0,0.0,1,False
7,tminus2_present,0.0,0.0,0.0,0.0,1,False
8,current_tminus1_cosine,0.0,0.0,0.0,0.0,1,False
9,current_tminus1_l1,0.0,0.0,0.0,0.0,1,False


In [42]:
# ============================================================
# 34. CLEAN TRAJECTORY ABLATION
#     Actual trajectory vs zero-history counterfactual
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

N_CLASSES = len(CLASS_NAMES)

oof_transition_actual_prob_clean = np.zeros(
    (len(y_train), N_CLASSES),
    dtype=np.float64,
)

oof_transition_zero_prob = np.zeros(
    (len(y_train), N_CLASSES),
    dtype=np.float64,
)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (tr_idx, va_idx) in enumerate(
    cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    # --------------------------------------------------------
    # Training data: real semantic + real trajectory
    # --------------------------------------------------------

    X_tr = np.hstack([
        X_sem_train[tr_idx],
        T_train[tr_idx],
    ])

    X_va_actual = np.hstack([
        X_sem_train[va_idx],
        T_train[va_idx],
    ])

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_actual_scaled = scaler.transform(X_va_actual)

    model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_tr_scaled,
        y_train[tr_idx],
    )

    # --------------------------------------------------------
    # ACTUAL trajectory
    # --------------------------------------------------------

    oof_transition_actual_prob_clean[
        va_idx
    ] = model.predict_proba(
        X_va_actual_scaled
    )

    # --------------------------------------------------------
    # COUNTERFACTUAL:
    # exact same semantic representation,
    # but no trajectory/history
    # --------------------------------------------------------

    T_va_zero = np.zeros_like(
        T_train[va_idx]
    )

    X_va_zero = np.hstack([
        X_sem_train[va_idx],
        T_va_zero,
    ])

    X_va_zero_scaled = scaler.transform(
        X_va_zero
    )

    oof_transition_zero_prob[
        va_idx
    ] = model.predict_proba(
        X_va_zero_scaled
    )

    print(f"Fold {fold} complete")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [43]:
# ============================================================
# 35. Clean causal trajectory effect
# ============================================================

pred_transition_actual_clean = (
    oof_transition_actual_prob_clean.argmax(axis=1)
)

pred_transition_zero = (
    oof_transition_zero_prob.argmax(axis=1)
)

actual_correct = (
    pred_transition_actual_clean == y_train
)

zero_correct = (
    pred_transition_zero == y_train
)

trajectory_rescue_clean = (
    (~zero_correct)
    &
    actual_correct
)

trajectory_break_clean = (
    zero_correct
    &
    (~actual_correct)
)


print("ALL EXAMPLES")
print("------------------------------")
print(
    "Trajectory rescues:",
    trajectory_rescue_clean.sum()
)
print(
    "Trajectory breaks:",
    trajectory_break_clean.sum()
)
print(
    "Net:",
    trajectory_rescue_clean.sum()
    - trajectory_break_clean.sum()
)

print()


# ============================================================
# Constraint-specific effect
# ============================================================

constraint_mask = (
    y_train == CONSTRAINT_ID
)

constraint_rescue_clean = (
    constraint_mask
    &
    trajectory_rescue_clean
)

constraint_break_clean = (
    constraint_mask
    &
    trajectory_break_clean
)

print("CONSTRAINT ERRORS")
print("------------------------------")

print(
    "Trajectory rescues:",
    constraint_rescue_clean.sum()
)

print(
    "Trajectory breaks:",
    constraint_break_clean.sum()
)

print(
    "Net:",
    constraint_rescue_clean.sum()
    - constraint_break_clean.sum()
)

ALL EXAMPLES
------------------------------
Trajectory rescues: 103
Trajectory breaks: 79
Net: 24

CONSTRAINT ERRORS
------------------------------
Trajectory rescues: 3
Trajectory breaks: 38
Net: -35


In [44]:
# ============================================================
# 36. Zero-history sanity check
# ============================================================

zero_history = (
    train_targets[
        "history_event_count"
    ].to_numpy()
    == 0
)

print(
    "Zero-history examples:",
    zero_history.sum()
)

print(
    "Prediction changes under ablation:",
    (
        pred_transition_actual_clean[
            zero_history
        ]
        !=
        pred_transition_zero[
            zero_history
        ]
    ).sum()
)

print(
    "Zero-history trajectory rescues:",
    (
        zero_history
        &
        trajectory_rescue_clean
    ).sum()
)

print(
    "Zero-history trajectory breaks:",
    (
        zero_history
        &
        trajectory_break_clean
    ).sum()
)

Zero-history examples: 56
Prediction changes under ablation: 0
Zero-history trajectory rescues: 0
Zero-history trajectory breaks: 0


In [45]:
# ============================================================
# 37. Actual trajectory vs no-history counterfactual
# ============================================================

comparison_clean = pd.DataFrame([
    {
        "model": "transition_zero_history",
        **metrics(pred_transition_zero),
    },
    {
        "model": "transition_actual_history",
        **metrics(pred_transition_actual_clean),
    },
])

display(
    comparison_clean.round(4)
)

print("\nMetric gains from trajectory:")

for metric_name in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:
    delta = (
        comparison_clean.loc[
            comparison_clean["model"]
            == "transition_actual_history",
            metric_name,
        ].iloc[0]
        -
        comparison_clean.loc[
            comparison_clean["model"]
            == "transition_zero_history",
            metric_name,
        ].iloc[0]
    )

    print(
        f"{metric_name:20s}: {delta:+.4f}"
    )

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,transition_zero_history,0.5191,0.4813,0.4824,0.5212
1,transition_actual_history,0.5353,0.4839,0.4981,0.5323



Metric gains from trajectory:
accuracy            : +0.0161
balanced_accuracy   : +0.0026
macro_f1            : +0.0157
weighted_f1         : +0.0111


In [46]:
# ============================================================
# 38. Trajectory causal utility by failure family
# ============================================================

rows = []

for class_id, family in enumerate(CLASS_NAMES):

    mask = (
        y_train == class_id
    )

    rescue = (
        mask
        &
        trajectory_rescue_clean
    ).sum()

    brk = (
        mask
        &
        trajectory_break_clean
    ).sum()

    support = mask.sum()

    zero_recall = (
        pred_transition_zero[mask]
        == y_train[mask]
    ).mean()

    actual_recall = (
        pred_transition_actual_clean[mask]
        == y_train[mask]
    ).mean()

    rows.append({
        "failure_family": family,
        "support": support,
        "zero_history_recall": zero_recall,
        "actual_history_recall": actual_recall,
        "recall_delta": (
            actual_recall
            - zero_recall
        ),
        "rescues": rescue,
        "breaks": brk,
        "net": rescue - brk,
        "rescue_rate": rescue / support,
        "break_rate": brk / support,
    })

trajectory_family_effect = (
    pd.DataFrame(rows)
    .sort_values(
        "net",
        ascending=False,
    )
)

display(
    trajectory_family_effect.round(4)
)

,failure_family,support,zero_history_recall,actual_history_recall,recall_delta,rescues,breaks,net,rescue_rate,break_rate
0,workflow_error,660,0.5576,0.6576,0.1000,72,6,66,0.1091,0.0091
3,grounding_state_error,244,0.3402,0.4467,0.1066,26,0,26,0.1066,0.0000
4,reasoning_value_error,31,0.3871,0.4516,0.0645,2,0,2,0.0645,0.0000
1,constraint_error,317,0.5521,0.4416,-0.1104,3,38,-35,0.0095,0.1199
2,tool_use_error,237,0.5696,0.4219,-0.1477,0,35,-35,0.0000,0.1477


Next research: where are those predictions going?

Before building another router, we need to understand the direction of the trajectory-induced class shifts.

Specifically:

When history destroys a correct constraint/tool-use prediction, which class does it push the model toward?

And conversely:

When history rescues workflow/grounding examples, where did the zero-history model put them?

In [47]:
# ============================================================
# 39. Trajectory-induced prediction flow matrix
# ============================================================

flow = pd.crosstab(
    pd.Series(
        [CLASS_NAMES[i] for i in pred_transition_zero],
        name="zero_history_prediction",
    ),
    pd.Series(
        [CLASS_NAMES[i] for i in pred_transition_actual_clean],
        name="actual_history_prediction",
    ),
)

display(flow)

actual_history_prediction,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
zero_history_prediction,,,,,
constraint_error,312,38,1,1,68
grounding_state_error,0,166,0,0,5
reasoning_value_error,0,1,22,0,0
tool_use_error,12,9,2,197,95
workflow_error,1,12,0,1,546


In [48]:
# ============================================================
# 40. Prediction flows caused by trajectory
# ============================================================

changed = (
    pred_transition_zero
    != pred_transition_actual_clean
)

print(
    "Predictions changed by trajectory:",
    changed.sum(),
)

print(
    "Change rate:",
    changed.mean(),
)

change_df = pd.DataFrame({
    "true_family": [
        CLASS_NAMES[i]
        for i in y_train[changed]
    ],
    "zero_prediction": [
        CLASS_NAMES[i]
        for i in pred_transition_zero[changed]
    ],
    "history_prediction": [
        CLASS_NAMES[i]
        for i in pred_transition_actual_clean[changed]
    ],
    "effect": np.where(
        trajectory_rescue_clean[changed],
        "rescue",
        np.where(
            trajectory_break_clean[changed],
            "break",
            "wrong_to_wrong",
        ),
    ),
})

display(
    change_df[
        "effect"
    ].value_counts()
)

display(
    change_df.groupby([
        "zero_prediction",
        "history_prediction",
        "effect",
    ])
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False,
    )
    .head(30)
)

Predictions changed by trajectory: 246
Change rate: 0.16521155137676294


effect
rescue            103
break              79
wrong_to_wrong     64
Name: count, dtype: int64

,zero_prediction,history_prediction,effect,count
20,tool_use_error,workflow_error,rescue,43
19,tool_use_error,workflow_error,break,31
5,constraint_error,workflow_error,break,28
6,constraint_error,workflow_error,rescue,26
21,tool_use_error,workflow_error,wrong_to_wrong,21
1,constraint_error,grounding_state_error,rescue,17
7,constraint_error,workflow_error,wrong_to_wrong,14
2,constraint_error,grounding_state_error,wrong_to_wrong,11
0,constraint_error,grounding_state_error,break,10
13,tool_use_error,constraint_error,wrong_to_wrong,7


In [49]:
# ============================================================
# 41. Where do broken constraint/tool-use predictions go?
# ============================================================

harmful_rows = []

for family in [
    "constraint_error",
    "tool_use_error",
]:

    family_id = CLASS_NAMES.index(family)

    mask = (
        (y_train == family_id)
        &
        trajectory_break_clean
    )

    for idx in np.where(mask)[0]:

        harmful_rows.append({
            "row_idx": idx,
            "true_family": family,
            "zero_prediction":
                CLASS_NAMES[
                    pred_transition_zero[idx]
                ],
            "history_prediction":
                CLASS_NAMES[
                    pred_transition_actual_clean[idx]
                ],
            "history_event_count":
                train_targets.iloc[idx][
                    "history_event_count"
                ],
        })


harmful_flow_df = pd.DataFrame(
    harmful_rows
)

display(
    harmful_flow_df.groupby([
        "true_family",
        "history_prediction",
    ])
    .size()
    .reset_index(name="count")
    .sort_values(
        [
            "true_family",
            "count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

,true_family,history_prediction,count
1,constraint_error,workflow_error,28
0,constraint_error,grounding_state_error,10
4,tool_use_error,workflow_error,31
2,tool_use_error,constraint_error,2
3,tool_use_error,grounding_state_error,2


In [50]:
# ============================================================
# 42. Where do workflow/grounding rescues come from?
# ============================================================

beneficial_rows = []

for family in [
    "workflow_error",
    "grounding_state_error",
]:

    family_id = CLASS_NAMES.index(family)

    mask = (
        (y_train == family_id)
        &
        trajectory_rescue_clean
    )

    for idx in np.where(mask)[0]:

        beneficial_rows.append({
            "row_idx": idx,
            "true_family": family,
            "zero_prediction":
                CLASS_NAMES[
                    pred_transition_zero[idx]
                ],
            "history_prediction":
                CLASS_NAMES[
                    pred_transition_actual_clean[idx]
                ],
            "history_event_count":
                train_targets.iloc[idx][
                    "history_event_count"
                ],
        })


beneficial_flow_df = pd.DataFrame(
    beneficial_rows
)

display(
    beneficial_flow_df.groupby([
        "true_family",
        "zero_prediction",
    ])
    .size()
    .reset_index(name="count")
    .sort_values(
        [
            "true_family",
            "count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

,true_family,zero_prediction,count
0,grounding_state_error,constraint_error,17
2,grounding_state_error,workflow_error,5
1,grounding_state_error,tool_use_error,4
5,workflow_error,tool_use_error,43
3,workflow_error,constraint_error,26
4,workflow_error,grounding_state_error,3


In [51]:
# ============================================================
# 43. Does trajectory systematically shift class predictions?
# ============================================================

distribution_rows = []

for class_id, family in enumerate(CLASS_NAMES):

    zero_count = (
        pred_transition_zero
        == class_id
    ).sum()

    actual_count = (
        pred_transition_actual_clean
        == class_id
    ).sum()

    distribution_rows.append({
        "failure_family": family,
        "zero_history_predictions":
            zero_count,
        "actual_history_predictions":
            actual_count,
        "delta":
            actual_count - zero_count,
        "zero_share":
            zero_count / len(y_train),
        "actual_share":
            actual_count / len(y_train),
    })


prediction_shift = pd.DataFrame(
    distribution_rows
)

display(
    prediction_shift
    .sort_values(
        "delta",
        ascending=False,
    )
    .round(4)
)

,failure_family,zero_history_predictions,actual_history_predictions,delta,zero_share,actual_share
0,workflow_error,560,714,154,0.3761,0.4795
3,grounding_state_error,171,226,55,0.1148,0.1518
4,reasoning_value_error,23,25,2,0.0154,0.0168
1,constraint_error,420,325,-95,0.2821,0.2183
2,tool_use_error,315,199,-116,0.2116,0.1336


The dominant trajectory effect is a systematic redistribution of probability/predictions:

workflow_error: 560 → 714, +154 predictions
grounding_state_error: 171 → 226, +55
constraint_error: 420 → 325, −95
tool_use_error: 315 → 199, −116
reasoning_value_error: essentially unchanged, +2

So trajectory context is not merely adding discriminative information. It creates a strong directional bias from constraint/tool-use → workflow/grounding. That bias is sometimes exactly right—e.g. 43 tool_use → workflow rescues and 17 constraint → grounding rescues—but it also explains most of the damage: 31 true tool-use cases are pushed to workflow, and 28 true constraint cases are pushed to workflow.

The next question should therefore be:

Can we retain trajectory's useful probability correction without allowing it to over-shift constraint/tool-use examples into workflow/grounding?

In [55]:
# ============================================================
# Trajectory block — no feature-name variable required
# ============================================================

print("X_sem_train:", X_sem_train.shape)
print("T_train:", T_train.shape)
print("X_trans_train:", X_trans_train.shape)

assert X_sem_train.shape[1] == 384
assert T_train.shape[1] == 16
assert X_trans_train.shape[1] == X_sem_train.shape[1] + T_train.shape[1]

trajectory_start = X_sem_train.shape[1]
trajectory_cols_idx = np.arange(
    trajectory_start,
    X_trans_train.shape[1]
)

print("Trajectory columns in X_trans_train:",
      trajectory_cols_idx[0], "to", trajectory_cols_idx[-1])

# Sanity check: transition tail should literally equal T_train
print(
    "Tail equals T_train:",
    np.allclose(
        X_trans_train[:, trajectory_start:],
        T_train
    )
)

X_sem_train: (1489, 384)
T_train: (1489, 16)
X_trans_train: (1489, 400)
Trajectory columns in X_trans_train: 384 to 399
Tail equals T_train: True


In [57]:
# ============================================================
# Find objects needed for trajectory-dose experiment
# ============================================================

for name, obj in list(globals().items()):
    try:
        if (
            "model" in name.lower()
            or "clf" in name.lower()
            or "coef" in name.lower()
            or "fold" in name.lower()
            or "transition" in name.lower()
        ):
            shape = getattr(obj, "shape", None)
            typ = type(obj).__name__

            if shape is not None:
                print(f"{name:45s} {typ:25s} {shape}")
            elif isinstance(obj, (list, tuple)):
                print(f"{name:45s} {typ:25s} len={len(obj)}")
    except Exception:
        pass

transition_names                              list                      len=16
transition_feature_df                         DataFrame                 (1489, 16)
constraint_transition_rank_all                ndarray                   (1489,)
oof_transition_actual_prob                    ndarray                   (1489, 5)
oof_transition_neutral_prob                   ndarray                   (1489, 5)
pred_transition_neutral                       ndarray                   (1489,)
pred_transition_actual                        ndarray                   (1489,)
oof_transition_actual_prob_clean              ndarray                   (1489, 5)
oof_transition_zero_prob                      ndarray                   (1489, 5)
pred_transition_actual_clean                  ndarray                   (1489,)
pred_transition_zero                          ndarray                   (1489,)


In [58]:
# ============================================================
# 44. Clean trajectory-strength interpolation
#     SAME fitted model, vary only trajectory contribution
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

alphas = [
    0.0,
    0.25,
    0.50,
    0.75,
    1.0,
]

N_CLASSES = len(CLASS_NAMES)

alpha_probabilities = {
    alpha: np.zeros(
        (len(y_train), N_CLASSES),
        dtype=np.float64,
    )
    for alpha in alphas
}


cv_alpha = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


for fold, (tr_idx, va_idx) in enumerate(
    cv_alpha.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    # --------------------------------------------------------
    # Train exactly the transition model used previously
    # --------------------------------------------------------

    X_tr = np.hstack([
        X_sem_train[tr_idx],
        T_train[tr_idx],
    ])

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(
        X_tr
    )

    model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_tr_scaled,
        y_train[tr_idx],
    )

    # --------------------------------------------------------
    # Same model, same scaler.
    # Only scale history at inference time.
    # --------------------------------------------------------

    for alpha in alphas:

        T_alpha = (
            T_train[va_idx]
            * alpha
        )

        X_va_alpha = np.hstack([
            X_sem_train[va_idx],
            T_alpha,
        ])

        X_va_alpha_scaled = (
            scaler.transform(
                X_va_alpha
            )
        )

        alpha_probabilities[
            alpha
        ][va_idx] = (
            model.predict_proba(
                X_va_alpha_scaled
            )
        )

    print(
        f"Fold {fold} complete"
    )


alpha_predictions = {
    alpha:
        prob.argmax(axis=1)

    for alpha, prob
    in alpha_probabilities.items()
}

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [59]:
# ============================================================
# 45. Endpoint validation
# ============================================================

print(
    "alpha=0 matches clean zero-history:",
    np.array_equal(
        alpha_predictions[0.0],
        pred_transition_zero,
    )
)

print(
    "alpha=1 matches actual history:",
    np.array_equal(
        alpha_predictions[1.0],
        pred_transition_actual_clean,
    )
)

print(
    "alpha=0 probability max diff:",
    np.abs(
        alpha_probabilities[0.0]
        -
        oof_transition_zero_prob
    ).max()
)

print(
    "alpha=1 probability max diff:",
    np.abs(
        alpha_probabilities[1.0]
        -
        oof_transition_actual_prob_clean
    ).max()
)

alpha=0 matches clean zero-history: True
alpha=1 matches actual history: True
alpha=0 probability max diff: 0.0
alpha=1 probability max diff: 0.0


In [60]:
# ============================================================
# 46. Overall trajectory dose-response
# ============================================================

alpha_rows = []

for alpha in alphas:

    pred = alpha_predictions[
        alpha
    ]

    result = metrics(
        pred
    )

    alpha_rows.append({
        "alpha": alpha,
        **result,

        "changed_vs_zero": int(
            (
                pred
                != alpha_predictions[0.0]
            ).sum()
        ),
    })


alpha_metrics_df = pd.DataFrame(
    alpha_rows
)

display(
    alpha_metrics_df.round(4)
)

,alpha,accuracy,balanced_accuracy,macro_f1,weighted_f1,changed_vs_zero
0,0.00,0.5191,0.4813,0.4824,0.5212,0
1,0.25,0.5245,0.4872,0.4905,0.5264,63
2,0.50,0.5299,0.4786,0.4871,0.5302,122
3,0.75,0.5366,0.4875,0.4976,0.5361,180
4,1.00,0.5353,0.4839,0.4981,0.5323,246


In [61]:
# ============================================================
# 47. Class-specific trajectory dose-response
# ============================================================

family_rows = []

for alpha in alphas:

    pred = alpha_predictions[
        alpha
    ]

    for class_id, family in enumerate(
        CLASS_NAMES
    ):

        mask = (
            y_train == class_id
        )

        family_rows.append({
            "alpha":
                alpha,

            "failure_family":
                family,

            "support":
                int(mask.sum()),

            "recall":
                (
                    pred[mask]
                    == class_id
                ).mean(),

            "prediction_count":
                int(
                    (
                        pred
                        == class_id
                    ).sum()
                ),
        })


alpha_family_df = pd.DataFrame(
    family_rows
)


print("RECALL")
display(
    alpha_family_df
    .pivot(
        index="failure_family",
        columns="alpha",
        values="recall",
    )
    .round(4)
)


print("\nPREDICTION COUNTS")
display(
    alpha_family_df
    .pivot(
        index="failure_family",
        columns="alpha",
        values="prediction_count",
    )
)

RECALL


alpha,0.00,0.25,0.50,0.75,1.00
failure_family,,,,,
constraint_error,0.5521,0.5331,0.5079,0.4763,0.4416
grounding_state_error,0.3402,0.3730,0.3893,0.4262,0.4467
reasoning_value_error,0.3871,0.4194,0.3871,0.4194,0.4516
tool_use_error,0.5696,0.5316,0.4979,0.4852,0.4219
workflow_error,0.5576,0.5788,0.6106,0.6303,0.6576



PREDICTION COUNTS


alpha,0.00,0.25,0.50,0.75,1.00
failure_family,,,,,
constraint_error,420,403,377,347,325
grounding_state_error,171,186,193,211,226
reasoning_value_error,23,25,24,25,25
tool_use_error,315,281,254,236,199
workflow_error,560,594,641,670,714


In [62]:
# ============================================================
# 48. Rescue/break curve by alpha
# ============================================================

zero_pred = (
    alpha_predictions[0.0]
)

zero_correct = (
    zero_pred
    == y_train
)


rows = []

for alpha in alphas:

    pred = (
        alpha_predictions[
            alpha
        ]
    )

    correct = (
        pred == y_train
    )

    rescues = (
        (~zero_correct)
        &
        correct
    )

    breaks = (
        zero_correct
        &
        (~correct)
    )

    rows.append({
        "alpha":
            alpha,

        "rescues":
            int(
                rescues.sum()
            ),

        "breaks":
            int(
                breaks.sum()
            ),

        "net":
            int(
                rescues.sum()
                -
                breaks.sum()
            ),

        "changed":
            int(
                (
                    pred
                    != zero_pred
                ).sum()
            ),
    })


alpha_rescue_df = pd.DataFrame(
    rows
)

display(
    alpha_rescue_df
)

,alpha,rescues,breaks,net,changed
0,0.00,0,0,0,0
1,0.25,25,17,8,63
2,0.50,51,35,16,122
3,0.75,78,52,26,180
4,1.00,103,79,24,246


In [63]:
# ============================================================
# 49. Class-specific causal utility by alpha
# ============================================================

rows = []

for alpha in alphas:

    pred = (
        alpha_predictions[
            alpha
        ]
    )

    correct = (
        pred == y_train
    )

    for class_id, family in enumerate(
        CLASS_NAMES
    ):

        mask = (
            y_train
            == class_id
        )

        rescue = (
            mask
            &
            (~zero_correct)
            &
            correct
        )

        brk = (
            mask
            &
            zero_correct
            &
            (~correct)
        )

        rows.append({
            "alpha":
                alpha,

            "failure_family":
                family,

            "rescues":
                int(
                    rescue.sum()
                ),

            "breaks":
                int(
                    brk.sum()
                ),

            "net":
                int(
                    rescue.sum()
                    -
                    brk.sum()
                ),
        })


alpha_family_effect_df = pd.DataFrame(
    rows
)


display(
    alpha_family_effect_df
    .pivot(
        index="failure_family",
        columns="alpha",
        values="net",
    )
)

alpha,0.00,0.25,0.50,0.75,1.00
failure_family,,,,,
constraint_error,0,-6,-14,-24,-35
grounding_state_error,0,8,12,21,26
reasoning_value_error,0,1,0,1,2
tool_use_error,0,-9,-17,-20,-35
workflow_error,0,14,35,48,66


In [64]:
# ============================================================
# 50. Approximate trajectory flip threshold
# ============================================================

fine_alphas = np.round(
    np.linspace(
        0.0,
        1.0,
        21,
    ),
    2,
)

fine_probabilities = {
    alpha: np.zeros(
        (len(y_train), N_CLASSES),
        dtype=np.float64,
    )
    for alpha in fine_alphas
}


cv_fine = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


for fold, (tr_idx, va_idx) in enumerate(
    cv_fine.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    X_tr = np.hstack([
        X_sem_train[tr_idx],
        T_train[tr_idx],
    ])

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(
        X_tr
    )

    model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_tr_scaled,
        y_train[tr_idx],
    )

    for alpha in fine_alphas:

        X_va = np.hstack([
            X_sem_train[va_idx],
            T_train[va_idx]
            * alpha,
        ])

        fine_probabilities[
            alpha
        ][va_idx] = (
            model.predict_proba(
                scaler.transform(
                    X_va
                )
            )
        )


fine_predictions = {
    alpha:
        prob.argmax(axis=1)

    for alpha, prob
    in fine_probabilities.items()
}


baseline_pred = (
    fine_predictions[0.0]
)

flip_alpha = np.full(
    len(y_train),
    np.nan,
)


for i in range(
    len(y_train)
):

    for alpha in fine_alphas[1:]:

        if (
            fine_predictions[
                alpha
            ][i]
            != baseline_pred[i]
        ):
            flip_alpha[i] = alpha
            break


print(
    pd.Series(
        flip_alpha
    ).describe()
)

print(
    "\nNever changed:",
    np.isnan(
        flip_alpha
    ).sum()
)

count    246.000000
mean       0.528252
std        0.294862
min        0.050000
25%        0.250000
50%        0.550000
75%        0.800000
max        1.000000
dtype: float64

Never changed: 1243


In [65]:
flip_df = pd.DataFrame({
    "true_family": [
        CLASS_NAMES[int(i)]
        for i in y_train
    ],

    "flip_alpha":
        flip_alpha,
})


display(
    flip_df
    .dropna()
    .groupby(
        "true_family"
    )[
        "flip_alpha"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
    .round(4)
)

,count,mean,median,std,min,max
true_family,,,,,,
constraint_error,54,0.5296,0.55,0.2921,0.05,1.0
grounding_state_error,45,0.4389,0.45,0.2751,0.05,1.0
reasoning_value_error,7,0.7000,0.75,0.2677,0.15,0.9
tool_use_error,45,0.5678,0.50,0.3237,0.05,1.0
workflow_error,95,0.5384,0.55,0.2874,0.05,1.0


In [66]:
# ============================================================
# 51. Trajectory utility conditioned on ZERO-HISTORY prediction
# ============================================================

zero_pred = alpha_predictions[0.0]
zero_correct = zero_pred == y_train

rows = []

for alpha in alphas:

    pred = alpha_predictions[alpha]
    correct = pred == y_train

    for source_id, source_name in enumerate(CLASS_NAMES):

        source_mask = zero_pred == source_id

        rescues = (
            source_mask
            & (~zero_correct)
            & correct
        )

        breaks = (
            source_mask
            & zero_correct
            & (~correct)
        )

        rows.append({
            "alpha": alpha,
            "zero_history_prediction": source_name,
            "support": int(source_mask.sum()),
            "rescues": int(rescues.sum()),
            "breaks": int(breaks.sum()),
            "net": int(rescues.sum() - breaks.sum()),
            "accuracy": (
                correct[source_mask].mean()
                if source_mask.sum() > 0
                else np.nan
            ),
            "change_rate": (
                (pred[source_mask] != zero_pred[source_mask]).mean()
                if source_mask.sum() > 0
                else np.nan
            ),
        })

source_alpha_df = pd.DataFrame(rows)

print("NET UTILITY BY ZERO-HISTORY PREDICTION")
display(
    source_alpha_df.pivot(
        index="zero_history_prediction",
        columns="alpha",
        values="net",
    )
)

print("\nACCURACY BY ZERO-HISTORY PREDICTION")
display(
    source_alpha_df.pivot(
        index="zero_history_prediction",
        columns="alpha",
        values="accuracy",
    ).round(4)
)

print("\nCHANGE RATE")
display(
    source_alpha_df.pivot(
        index="zero_history_prediction",
        columns="alpha",
        values="change_rate",
    ).round(4)
)

NET UTILITY BY ZERO-HISTORY PREDICTION


alpha,0.00,0.25,0.50,0.75,1.00
zero_history_prediction,,,,,
constraint_error,0,2,4,9,6
grounding_state_error,0,1,2,2,3
reasoning_value_error,0,0,0,0,0
tool_use_error,0,3,9,17,16
workflow_error,0,2,1,-2,-1



ACCURACY BY ZERO-HISTORY PREDICTION


alpha,0.00,0.25,0.50,0.75,1.00
zero_history_prediction,,,,,
constraint_error,0.4167,0.4214,0.4262,0.4381,0.4310
grounding_state_error,0.4854,0.4912,0.4971,0.4971,0.5029
reasoning_value_error,0.5217,0.5217,0.5217,0.5217,0.5217
tool_use_error,0.4286,0.4381,0.4571,0.4825,0.4794
workflow_error,0.6571,0.6607,0.6589,0.6536,0.6554



CHANGE RATE


alpha,0.00,0.25,0.50,0.75,1.00
zero_history_prediction,,,,,
constraint_error,0.0,0.0524,0.1214,0.1952,0.2571
grounding_state_error,0.0,0.0058,0.0175,0.0234,0.0292
reasoning_value_error,0.0,0.0000,0.0000,0.0000,0.0435
tool_use_error,0.0,0.1111,0.1968,0.2571,0.3746
workflow_error,0.0,0.0089,0.0107,0.0232,0.0250


In [67]:
# ============================================================
# 52. Best alpha by zero-history predicted family
# ============================================================

best_source_alpha = (
    source_alpha_df
    .sort_values(
        [
            "zero_history_prediction",
            "accuracy",
            "net",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    .groupby(
        "zero_history_prediction",
        as_index=False,
    )
    .first()
)

display(
    best_source_alpha[
        [
            "zero_history_prediction",
            "support",
            "alpha",
            "accuracy",
            "rescues",
            "breaks",
            "net",
            "change_rate",
        ]
    ].round(4)
)

,zero_history_prediction,support,alpha,accuracy,rescues,breaks,net,change_rate
0,constraint_error,420,0.75,0.4381,35,26,9,0.1952
1,grounding_state_error,171,1.00,0.5029,3,0,3,0.0292
2,reasoning_value_error,23,0.00,0.5217,0,0,0,0.0000
3,tool_use_error,315,0.75,0.4825,37,20,17,0.2571
4,workflow_error,560,0.25,0.6607,3,1,2,0.0089


In [68]:
# ============================================================
# 53. Diagnostic source-conditional alpha policy
#
# IMPORTANT:
# This is selected/evaluated on the same OOF data.
# Treat as an optimistic diagnostic, NOT final performance.
# ============================================================

source_alpha_map = dict(
    zip(
        best_source_alpha[
            "zero_history_prediction"
        ],
        best_source_alpha[
            "alpha"
        ],
    )
)

print("Selected alpha policy:")
for family in CLASS_NAMES:
    print(
        f"{family:25s}",
        source_alpha_map[family]
    )


conditional_pred = np.empty_like(
    zero_pred
)

for source_id, source_name in enumerate(
    CLASS_NAMES
):

    mask = (
        zero_pred == source_id
    )

    alpha = source_alpha_map[
        source_name
    ]

    conditional_pred[mask] = (
        alpha_predictions[
            alpha
        ][mask]
    )


conditional_metrics = metrics(
    conditional_pred
)

print(
    "\nSource-conditional metrics:"
)

print(
    conditional_metrics
)


conditional_correct = (
    conditional_pred
    == y_train
)

rescues = (
    (~zero_correct)
    &
    conditional_correct
)

breaks = (
    zero_correct
    &
    (~conditional_correct)
)

print(
    "\nRescues:",
    rescues.sum()
)

print(
    "Breaks:",
    breaks.sum()
)

print(
    "Net:",
    rescues.sum()
    -
    breaks.sum()
)

Selected alpha policy:
workflow_error            0.25
constraint_error          0.75
tool_use_error            0.75
grounding_state_error     1.0
reasoning_value_error     0.0

Source-conditional metrics:
{'accuracy': 0.5399597044996642, 'balanced_accuracy': 0.48849053743229165, 'macro_f1': 0.4994443385321117, 'weighted_f1': 0.5389413677096767}

Rescues: 78
Breaks: 47
Net: 31


In [69]:
# ============================================================
# 54. Cross-fitted source-conditional trajectory policy
#
# Select best alpha separately for each zero-history predicted
# family using TRAIN folds only, then apply to held-out fold.
#
# This tests whether adaptive trajectory strength generalizes.
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

alphas = sorted(alpha_predictions.keys())

zero_pred = alpha_predictions[0.0]
zero_correct = zero_pred == y_train

cv_policy = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=123,   # separate policy-selection partition
)

crossfit_conditional_pred = np.empty_like(y_train)

policy_rows = []


for fold, (tr_idx, va_idx) in enumerate(
    cv_policy.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    fold_policy = {}

    # --------------------------------------------------------
    # Select alpha using TRAIN portion only
    # --------------------------------------------------------

    for source_id, source_name in enumerate(CLASS_NAMES):

        source_train = (
            zero_pred[tr_idx] == source_id
        )

        candidate_rows = []

        for alpha in alphas:

            pred_alpha_train = (
                alpha_predictions[alpha][tr_idx]
            )

            if source_train.sum() == 0:
                acc = np.nan
            else:
                acc = (
                    pred_alpha_train[source_train]
                    == y_train[tr_idx][source_train]
                ).mean()

            candidate_rows.append({
                "alpha": alpha,
                "accuracy": acc,
            })

        candidate_df = pd.DataFrame(candidate_rows)

        # Prefer highest conditional accuracy.
        # On exact ties, prefer smaller alpha:
        # conservative use of trajectory.
        best = (
            candidate_df
            .sort_values(
                ["accuracy", "alpha"],
                ascending=[False, True],
            )
            .iloc[0]
        )

        best_alpha = float(best["alpha"])

        fold_policy[source_id] = best_alpha

        policy_rows.append({
            "fold": fold,
            "source_id": source_id,
            "zero_history_prediction": source_name,
            "train_support": int(source_train.sum()),
            "selected_alpha": best_alpha,
            "train_accuracy": best["accuracy"],
        })

    # --------------------------------------------------------
    # Apply selected source-specific alpha to held-out fold
    # --------------------------------------------------------

    for source_id, source_name in enumerate(CLASS_NAMES):

        mask_local = (
            zero_pred[va_idx] == source_id
        )

        alpha = fold_policy[source_id]

        crossfit_conditional_pred[
            va_idx[mask_local]
        ] = (
            alpha_predictions[alpha][
                va_idx[mask_local]
            ]
        )

    print(
        f"Fold {fold}:",
        {
            CLASS_NAMES[k]: v
            for k, v in fold_policy.items()
        }
    )


policy_df = pd.DataFrame(policy_rows)

Fold 1: {'workflow_error': 0.0, 'constraint_error': 0.75, 'tool_use_error': 0.75, 'grounding_state_error': 1.0, 'reasoning_value_error': 0.0}
Fold 2: {'workflow_error': 0.25, 'constraint_error': 0.75, 'tool_use_error': 0.75, 'grounding_state_error': 1.0, 'reasoning_value_error': 0.0}
Fold 3: {'workflow_error': 0.25, 'constraint_error': 1.0, 'tool_use_error': 1.0, 'grounding_state_error': 1.0, 'reasoning_value_error': 0.0}
Fold 4: {'workflow_error': 0.25, 'constraint_error': 0.75, 'tool_use_error': 0.75, 'grounding_state_error': 1.0, 'reasoning_value_error': 0.0}
Fold 5: {'workflow_error': 0.25, 'constraint_error': 0.0, 'tool_use_error': 0.75, 'grounding_state_error': 0.5, 'reasoning_value_error': 0.0}


In [70]:
# ============================================================
# 55. Cross-fitted adaptive-alpha performance
# ============================================================

crossfit_metrics = metrics(
    crossfit_conditional_pred
)

crossfit_correct = (
    crossfit_conditional_pred == y_train
)

rescues = (
    (~zero_correct)
    & crossfit_correct
)

breaks = (
    zero_correct
    & (~crossfit_correct)
)

print("Cross-fitted adaptive alpha:")
print(crossfit_metrics)

print("\nRescues:", rescues.sum())
print("Breaks:", breaks.sum())
print("Net:", rescues.sum() - breaks.sum())


comparison = pd.DataFrame([
    {
        "model": "trajectory_alpha_0",
        **metrics(alpha_predictions[0.0]),
    },
    {
        "model": "trajectory_alpha_0.25",
        **metrics(alpha_predictions[0.25]),
    },
    {
        "model": "trajectory_alpha_0.50",
        **metrics(alpha_predictions[0.50]),
    },
    {
        "model": "trajectory_alpha_0.75",
        **metrics(alpha_predictions[0.75]),
    },
    {
        "model": "trajectory_alpha_1.00",
        **metrics(alpha_predictions[1.0]),
    },
    {
        "model": "adaptive_alpha_diagnostic",
        **metrics(conditional_pred),
    },
    {
        "model": "adaptive_alpha_crossfit",
        **crossfit_metrics,
    },
])

display(
    comparison.sort_values(
        "accuracy",
        ascending=False,
    ).round(4)
)

Cross-fitted adaptive alpha:
{'accuracy': 0.5231699126930826, 'balanced_accuracy': 0.470528603344913, 'macro_f1': 0.48242638393246284, 'weighted_f1': 0.5214291641899769}

Rescues: 64
Breaks: 58
Net: 6


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
5,adaptive_alpha_diagnostic,0.5400,0.4885,0.4994,0.5389
3,trajectory_alpha_0.75,0.5366,0.4875,0.4976,0.5361
4,trajectory_alpha_1.00,0.5353,0.4839,0.4981,0.5323
2,trajectory_alpha_0.50,0.5299,0.4786,0.4871,0.5302
1,trajectory_alpha_0.25,0.5245,0.4872,0.4905,0.5264
6,adaptive_alpha_crossfit,0.5232,0.4705,0.4824,0.5214
0,trajectory_alpha_0,0.5191,0.4813,0.4824,0.5212


In [71]:
# ============================================================
# 56. Is the selected alpha stable?
# ============================================================

print("SELECTED ALPHA BY FOLD")

display(
    policy_df.pivot(
        index="zero_history_prediction",
        columns="fold",
        values="selected_alpha",
    )
)


print("\nALPHA FREQUENCY")

display(
    policy_df.groupby(
        [
            "zero_history_prediction",
            "selected_alpha",
        ]
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values(
        [
            "zero_history_prediction",
            "count",
        ],
        ascending=[True, False],
    )
)


print("\nMEAN SELECTED ALPHA")

display(
    policy_df.groupby(
        "zero_history_prediction"
    )
    .agg(
        mean_alpha=(
            "selected_alpha",
            "mean",
        ),
        median_alpha=(
            "selected_alpha",
            "median",
        ),
        min_alpha=(
            "selected_alpha",
            "min",
        ),
        max_alpha=(
            "selected_alpha",
            "max",
        ),
        mean_train_support=(
            "train_support",
            "mean",
        ),
    )
    .round(4)
)

SELECTED ALPHA BY FOLD


fold,1,2,3,4,5
zero_history_prediction,,,,,
constraint_error,0.75,0.75,1.00,0.75,0.00
grounding_state_error,1.00,1.00,1.00,1.00,0.50
reasoning_value_error,0.00,0.00,0.00,0.00,0.00
tool_use_error,0.75,0.75,1.00,0.75,0.75
workflow_error,0.00,0.25,0.25,0.25,0.25



ALPHA FREQUENCY


,zero_history_prediction,selected_alpha,count
1,constraint_error,0.75,3
0,constraint_error,0.00,1
2,constraint_error,1.00,1
4,grounding_state_error,1.00,4
3,grounding_state_error,0.50,1
5,reasoning_value_error,0.00,5
6,tool_use_error,0.75,4
7,tool_use_error,1.00,1
9,workflow_error,0.25,4
8,workflow_error,0.00,1



MEAN SELECTED ALPHA


,mean_alpha,median_alpha,min_alpha,max_alpha,mean_train_support
zero_history_prediction,,,,,
constraint_error,0.65,0.75,0.00,1.00,336.0
grounding_state_error,0.90,1.00,0.50,1.00,136.8
reasoning_value_error,0.00,0.00,0.00,0.00,18.4
tool_use_error,0.80,0.75,0.75,1.00,252.0
workflow_error,0.20,0.25,0.00,0.25,448.0


In [72]:
# ============================================================
# 57. Inspect trajectory feature groups
# ============================================================

print("Trajectory features:", len(transition_names))

for i, name in enumerate(transition_names):
    print(f"{i:2d}: {name}")

Trajectory features: 16
 0: current_tminus3_cosine
 1: current_tminus3_l1
 2: current_tminus3_l2
 3: tminus3_present
 4: current_tminus2_cosine
 5: current_tminus2_l1
 6: current_tminus2_l2
 7: tminus2_present
 8: current_tminus1_cosine
 9: current_tminus1_l1
10: current_tminus1_l2
11: tminus1_present
12: history_transition_1_cosine
13: history_transition_1_l2
14: history_transition_2_cosine
15: history_transition_2_l2


In [73]:
# ============================================================
# 58. Build trajectory feature groups
# ============================================================

trajectory_groups = {
    "current_cosine": [
        i for i, name in enumerate(transition_names)
        if "current_" in name
        and "cosine" in name
    ],

    "current_distance": [
        i for i, name in enumerate(transition_names)
        if "current_" in name
        and (
            "_l1" in name
            or "_l2" in name
        )
    ],

    "history_presence": [
        i for i, name in enumerate(transition_names)
        if "present" in name
    ],

    "history_transition_cosine": [
        i for i, name in enumerate(transition_names)
        if "history_transition" in name
        and "cosine" in name
    ],

    "history_transition_distance": [
        i for i, name in enumerate(transition_names)
        if "history_transition" in name
        and "_l2" in name
    ],
}

for group, idx in trajectory_groups.items():

    print(f"\n{group} ({len(idx)} features)")

    for i in idx:
        print("   ", transition_names[i])


current_cosine (3 features)
    current_tminus3_cosine
    current_tminus2_cosine
    current_tminus1_cosine

current_distance (6 features)
    current_tminus3_l1
    current_tminus3_l2
    current_tminus2_l1
    current_tminus2_l2
    current_tminus1_l1
    current_tminus1_l2

history_presence (3 features)
    tminus3_present
    tminus2_present
    tminus1_present

history_transition_cosine (2 features)
    history_transition_1_cosine
    history_transition_2_cosine

history_transition_distance (2 features)
    history_transition_1_l2
    history_transition_2_l2


In [74]:
# ============================================================
# 59. Verify grouping
# ============================================================

all_grouped = [
    i
    for idx in trajectory_groups.values()
    for i in idx
]

print("Grouped feature count:", len(all_grouped))
print("Unique grouped count:", len(set(all_grouped)))
print("Expected:", len(transition_names))

missing = sorted(
    set(range(len(transition_names)))
    - set(all_grouped)
)

duplicates = sorted({
    i
    for i in all_grouped
    if all_grouped.count(i) > 1
})

print("\nMissing:")
for i in missing:
    print(i, transition_names[i])

print("\nDuplicates:")
for i in duplicates:
    print(i, transition_names[i])

Grouped feature count: 16
Unique grouped count: 16
Expected: 16

Missing:

Duplicates:


In [75]:
# ============================================================
# 60. Cross-fitted trajectory-group ablation
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

ablation_sets = {
    "all_trajectory": [],
    **{
        f"remove_{group}": idx
        for group, idx in trajectory_groups.items()
    },
    "zero_trajectory": list(
        range(T_train.shape[1])
    ),
}

ablation_predictions = {}
ablation_probabilities = {}


cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


for experiment, remove_idx in ablation_sets.items():

    print("\n", experiment)

    oof_prob = np.zeros(
        (len(y_train), len(CLASS_NAMES)),
        dtype=float,
    )

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X_sem_train,
            y_train,
            groups=groups_train,
        ),
        start=1,
    ):

        T_tr = T_train[tr_idx].copy()
        T_va = T_train[va_idx].copy()

        if len(remove_idx) > 0:
            T_tr[:, remove_idx] = 0.0
            T_va[:, remove_idx] = 0.0

        X_tr = np.hstack([
            X_sem_train[tr_idx],
            T_tr,
        ])

        X_va = np.hstack([
            X_sem_train[va_idx],
            T_va,
        ])

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_va_scaled = scaler.transform(
            X_va
        )

        model = LogisticRegression(
            C=0.01,
            max_iter=5000,
            random_state=42,
        )

        model.fit(
            X_tr_scaled,
            y_train[tr_idx],
        )

        oof_prob[va_idx] = (
            model.predict_proba(
                X_va_scaled
            )
        )

    ablation_probabilities[
        experiment
    ] = oof_prob

    ablation_predictions[
        experiment
    ] = oof_prob.argmax(axis=1)


 all_trajectory

 remove_current_cosine

 remove_current_distance

 remove_history_presence

 remove_history_transition_cosine

 remove_history_transition_distance

 zero_trajectory


In [76]:
# ============================================================
# 61. Group-ablation results
# ============================================================

rows = []

full_pred = ablation_predictions[
    "all_trajectory"
]

full_correct = (
    full_pred == y_train
)

for experiment, pred in ablation_predictions.items():

    result = metrics(pred)

    correct = (
        pred == y_train
    )

    rows.append({
        "experiment": experiment,
        **result,

        "delta_accuracy_vs_full":
            result["accuracy"]
            - metrics(full_pred)["accuracy"],

        "changed_vs_full":
            int((pred != full_pred).sum()),

        "rescues_vs_full":
            int(
                (
                    (~full_correct)
                    & correct
                ).sum()
            ),

        "breaks_vs_full":
            int(
                (
                    full_correct
                    & (~correct)
                ).sum()
            ),
    })


ablation_results_df = (
    pd.DataFrame(rows)
    .sort_values(
        "accuracy",
        ascending=False,
    )
)

display(
    ablation_results_df.round(4)
)

,experiment,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_full,changed_vs_full,rescues_vs_full,breaks_vs_full
1,remove_current_cosine,0.5413,0.4875,0.5021,0.5381,0.0060,19,12,3
3,remove_history_presence,0.5366,0.4852,0.4997,0.5337,0.0013,10,5,3
4,remove_history_transition_cosine,0.5366,0.4849,0.4990,0.5338,0.0013,17,7,5
0,all_trajectory,0.5353,0.4839,0.4981,0.5323,0.0000,0,0,0
5,remove_history_transition_distance,0.5353,0.4839,0.4982,0.5323,0.0000,5,2,2
2,remove_current_distance,0.5339,0.4745,0.4896,0.5303,-0.0013,42,16,18
6,zero_trajectory,0.5299,0.4784,0.4914,0.5264,-0.0054,114,36,44


The strongest observation is that trajectory is useful overall, but not all trajectory features are useful. In particular, the three current_tminus{k}_cosine features appear actively harmful: removing them raises accuracy from 0.5353 → 0.5413, macro-F1 from 0.4981 → 0.5021, with 12 rescues vs only 3 breaks. That's the cleanest ablation result so far.

By contrast, current_distance appears to contain useful signal. Removing its six L1/L2 features drops accuracy to 0.5339, balanced accuracy to 0.4745, and macro-F1 to 0.4896. The historical-transition distance features appear almost irrelevant, while presence and historical-transition cosine are slightly harmful.

So the working interpretation should now be:

The value of trajectory context comes primarily from embedding-distance information, not cosine similarity. Current-to-history cosine features introduce noise or a misleading notion of similarity, while L1/L2 distances carry the useful trajectory signal.

There is one caveat: these are correlated feature groups and retraining ablations, so we should not interpret the deltas as additive causal contributions. But the current_cosine result is large enough to justify the next experiment.

Next: construct a reduced trajectory representation

Rather than continuing one-group-at-a-time ablations, test nested representations. This tells us whether we can get a simpler and better transition model.

In [77]:
# ============================================================
# 62. Reduced trajectory representations
# ============================================================

group = trajectory_groups

feature_sets = {
    # Baselines
    "zero_trajectory": [],

    "all_trajectory": list(range(16)),

    # Individual components
    "current_distance_only":
        group["current_distance"],

    "current_cosine_only":
        group["current_cosine"],

    "presence_only":
        group["history_presence"],

    "history_transition_only":
        group["history_transition_cosine"]
        + group["history_transition_distance"],

    # Main reduced candidates
    "current_distance_plus_presence":
        group["current_distance"]
        + group["history_presence"],

    "current_distance_plus_history_transition":
        group["current_distance"]
        + group["history_transition_cosine"]
        + group["history_transition_distance"],

    "current_distance_plus_presence_plus_history_transition":
        group["current_distance"]
        + group["history_presence"]
        + group["history_transition_cosine"]
        + group["history_transition_distance"],

    # Everything except harmful current cosine
    "all_except_current_cosine": [
        i for i in range(16)
        if i not in group["current_cosine"]
    ],
}

for name, idx in feature_sets.items():
    print(f"{name:50s} {len(idx):2d} features")

zero_trajectory                                     0 features
all_trajectory                                     16 features
current_distance_only                               6 features
current_cosine_only                                 3 features
presence_only                                       3 features
history_transition_only                             4 features
current_distance_plus_presence                      9 features
current_distance_plus_history_transition           10 features
current_distance_plus_presence_plus_history_transition 13 features
all_except_current_cosine                          13 features


In [78]:
# ============================================================
# 63. Cross-fitted reduced trajectory models
# ============================================================

representation_predictions = {}
representation_probabilities = {}

for experiment, keep_idx in feature_sets.items():

    print("\n", experiment)

    oof_prob = np.zeros(
        (len(y_train), len(CLASS_NAMES)),
        dtype=float,
    )

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X_sem_train,
            y_train,
            groups=groups_train,
        ),
        start=1,
    ):

        # Semantic representation always retained.
        if len(keep_idx) > 0:

            X_tr = np.hstack([
                X_sem_train[tr_idx],
                T_train[tr_idx][:, keep_idx],
            ])

            X_va = np.hstack([
                X_sem_train[va_idx],
                T_train[va_idx][:, keep_idx],
            ])

        else:
            X_tr = X_sem_train[tr_idx]
            X_va = X_sem_train[va_idx]

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)

        model = LogisticRegression(
            C=0.01,
            max_iter=5000,
            random_state=42,
        )

        model.fit(
            X_tr_scaled,
            y_train[tr_idx],
        )

        oof_prob[va_idx] = model.predict_proba(
            X_va_scaled
        )

    representation_probabilities[
        experiment
    ] = oof_prob

    representation_predictions[
        experiment
    ] = oof_prob.argmax(axis=1)


 zero_trajectory

 all_trajectory

 current_distance_only

 current_cosine_only

 presence_only

 history_transition_only

 current_distance_plus_presence

 current_distance_plus_history_transition

 current_distance_plus_presence_plus_history_transition

 all_except_current_cosine


In [79]:
# ============================================================
# 64. Reduced representation results
# ============================================================

rows = []

baseline_pred = representation_predictions[
    "zero_trajectory"
]

baseline_correct = baseline_pred == y_train

for experiment, pred in representation_predictions.items():

    result = metrics(pred)
    correct = pred == y_train

    rows.append({
        "experiment": experiment,
        "n_trajectory_features":
            len(feature_sets[experiment]),

        **result,

        "delta_accuracy_vs_zero":
            result["accuracy"]
            - metrics(baseline_pred)["accuracy"],

        "rescues_vs_zero":
            int(
                ((~baseline_correct) & correct).sum()
            ),

        "breaks_vs_zero":
            int(
                (baseline_correct & (~correct)).sum()
            ),

        "net_vs_zero":
            int(
                ((~baseline_correct) & correct).sum()
                -
                (baseline_correct & (~correct)).sum()
            ),

        "changed_vs_zero":
            int((pred != baseline_pred).sum()),
    })


representation_results_df = (
    pd.DataFrame(rows)
    .sort_values(
        ["accuracy", "macro_f1"],
        ascending=False,
    )
)

display(
    representation_results_df.round(4)
)

,experiment,n_trajectory_features,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_zero,rescues_vs_zero,breaks_vs_zero,net_vs_zero,changed_vs_zero
6,current_distance_plus_presence,9,0.5420,0.4885,0.5028,0.5390,0.0128,46,27,19,100
7,current_distance_plus_history_transition,10,0.5413,0.4874,0.5024,0.5379,0.0121,46,28,18,100
8,current_distance_plus_presence_plus_history_tr...,13,0.5413,0.4875,0.5021,0.5381,0.0121,47,29,18,107
9,all_except_current_cosine,13,0.5413,0.4875,0.5021,0.5381,0.0121,47,29,18,107
3,current_cosine_only,3,0.5386,0.4796,0.4917,0.5353,0.0094,38,24,14,80
2,current_distance_only,6,0.5379,0.4983,0.5105,0.5352,0.0087,34,21,13,70
1,all_trajectory,16,0.5353,0.4839,0.4981,0.5323,0.0060,45,36,9,115
4,presence_only,3,0.5326,0.4824,0.4965,0.5298,0.0034,15,10,5,46
0,zero_trajectory,0,0.5292,0.4777,0.4908,0.5257,0.0000,0,0,0,0
5,history_transition_only,4,0.5292,0.4707,0.4846,0.5255,0.0000,18,18,0,60


In [80]:
# ============================================================
# 65. Family-level utility of reduced representations
# ============================================================

from sklearn.metrics import recall_score

rows = []

for experiment, pred in representation_predictions.items():

    for class_idx, class_name in enumerate(CLASS_NAMES):

        mask = y_train == class_idx

        rows.append({
            "experiment": experiment,
            "failure_family": class_name,
            "support": int(mask.sum()),
            "recall": float(
                (pred[mask] == class_idx).mean()
            ),
        })


family_recall_df = pd.DataFrame(rows)

family_recall_pivot = family_recall_df.pivot(
    index="failure_family",
    columns="experiment",
    values="recall",
)

display(
    family_recall_pivot.round(4)
)

experiment,all_except_current_cosine,all_trajectory,current_cosine_only,current_distance_only,current_distance_plus_history_transition,current_distance_plus_presence,current_distance_plus_presence_plus_history_transition,history_transition_only,presence_only,zero_trajectory
failure_family,,,,,,,,,,
constraint_error,0.4448,0.4416,0.4448,0.4448,0.4543,0.4511,0.4448,0.4353,0.4385,0.4290
grounding_state_error,0.4467,0.4467,0.4467,0.4385,0.4385,0.4467,0.4467,0.4262,0.4385,0.4385
reasoning_value_error,0.4516,0.4516,0.4194,0.5161,0.4516,0.4516,0.4516,0.4194,0.4516,0.4516
tool_use_error,0.4262,0.4219,0.4219,0.4346,0.4262,0.4262,0.4262,0.4135,0.4304,0.4135
workflow_error,0.6682,0.6576,0.6652,0.6576,0.6667,0.6667,0.6682,0.6591,0.6530,0.6561


This is a much cleaner result. The reduced-representation experiment confirms that the trajectory block should not be treated as a single feature family.

What we learned

The best accuracy comes from current_distance + presence:

Representation	# traj. features	Accuracy	Balanced acc.	Macro-F1	Net vs zero
current distance + presence	9	0.5420	0.4885	0.5028	+19
current distance only	6	0.5379	0.4983	0.5105	+13
all except current cosine	13	0.5413	0.4875	0.5021	+18
all trajectory	16	0.5353	0.4839	0.4981	+9
zero trajectory	0	0.5292	0.4777	0.4908	—

This gives us two different winners, depending on the objective.

current_distance + presence is the accuracy winner, reaching 54.20%, versus 52.92% with no trajectory and 53.53% with the full trajectory representation.

But current_distance_only is the class-balance winner. Its accuracy is slightly lower at 53.79%, but its balanced accuracy reaches 49.83% and macro-F1 51.05%, both substantially better than the full trajectory model.

That distinction matters because this is an imbalanced five-family problem. I would not automatically choose the 54.20% model.

The more important discovery: distance alone is remarkably good

The six current-distance features give:

accuracy: 0.5379
balanced accuracy: 0.4983
macro-F1: 0.5105
34 rescues / 21 breaks
net +13

And the family recalls are unusually balanced:

constraint: 0.4448
grounding: 0.4385
reasoning: 0.5161
tool use: 0.4346
workflow: 0.6576

This is actually more interesting than the 0.5420 accuracy result.

Recall the earlier problem: trajectory was improving workflow while sacrificing constraint and tool-use. The reduced distance representation substantially moderates that behavior.

In particular, reasoning_value_error goes from 0.4516 → 0.5161, although support is only 31, so we shouldn't overinterpret that class yet.

Presence is mostly an accuracy/bias adjustment

Compare:

distance only
→ 0.5379 accuracy / 0.5105 macro-F1

distance + presence
→ 0.5420 accuracy / 0.5028 macro-F1

Presence buys about +0.41 percentage points accuracy, but costs about -0.77 points macro-F1.

It shifts recall toward workflow:

workflow: 0.6576 → 0.6667
constraint: 0.4448 → 0.4511
grounding: 0.4385 → 0.4467
tool: 0.4346 → 0.4262
reasoning: 0.5161 → 0.4516

So presence isn't simply "more useful trajectory information." It changes the operating point.

History-transition features don't justify their complexity

history_transition_only gives exactly the same accuracy as zero trajectory:

0.5292 vs 0.5292

but balanced accuracy falls:

0.4777 → 0.4707

and macro-F1:

0.4908 → 0.4846

It produces exactly 18 rescues and 18 breaks.

Even when added to current distance:

distance only: 0.5379 / 0.4983 / 0.5105
distance + transition history: 0.5413 / 0.4874 / 0.5024

Again we get more raw accuracy but worse balanced performance.

This reinforces a broader pattern: longer-range trajectory structure isn't giving us robust class-discriminative information.

Current cosine needs a more nuanced conclusion

Our previous ablation suggested current cosine was simply harmful. The new experiment shows that's slightly too strong.

current_cosine_only actually achieves 0.5386 accuracy, which is better than zero trajectory and even slightly higher than distance-only accuracy.

But:

balanced accuracy = 0.4796
macro-F1 = 0.4917

So cosine isn't devoid of signal. Rather, it appears to contain class-skewed signal.

Look at workflow recall:

zero: 0.6561
cosine only: 0.6652

while tool-use is only 0.4219.

Therefore I'd revise the conclusion to:

Current cosine similarity contains predictive information, but much of its value appears aligned with the dominant workflow class. Distance features encode a more balanced and transferable trajectory signal.

And when cosine and distance coexist, the cosine block appears to interfere with the better distance representation through the jointly regularized classifier.

Next experiment: decompose L1 vs L2 and temporal lag

We now know that the six current-distance features are the strongest compact representation. But we still don't know why.

Those six features are:

t-3 L1
t-3 L2
t-2 L1
t-2 L2
t-1 L1
t-1 L2

There are two questions:

Is L1 or L2 actually carrying the signal?
Does recent history (t-1) matter more than older history (t-2, t-3)?

In [81]:
# ============================================================
# 66. Fine-grained current-distance decomposition
# ============================================================

name_to_idx = {
    name: i
    for i, name in enumerate(transition_names)
}

distance_feature_sets = {
    "zero": [],

    "all_current_distance": [
        name_to_idx["current_tminus3_l1"],
        name_to_idx["current_tminus3_l2"],
        name_to_idx["current_tminus2_l1"],
        name_to_idx["current_tminus2_l2"],
        name_to_idx["current_tminus1_l1"],
        name_to_idx["current_tminus1_l2"],
    ],

    # Metric decomposition
    "l1_only": [
        name_to_idx["current_tminus3_l1"],
        name_to_idx["current_tminus2_l1"],
        name_to_idx["current_tminus1_l1"],
    ],

    "l2_only": [
        name_to_idx["current_tminus3_l2"],
        name_to_idx["current_tminus2_l2"],
        name_to_idx["current_tminus1_l2"],
    ],

    # Lag decomposition
    "tminus1_only": [
        name_to_idx["current_tminus1_l1"],
        name_to_idx["current_tminus1_l2"],
    ],

    "tminus2_only": [
        name_to_idx["current_tminus2_l1"],
        name_to_idx["current_tminus2_l2"],
    ],

    "tminus3_only": [
        name_to_idx["current_tminus3_l1"],
        name_to_idx["current_tminus3_l2"],
    ],

    # Cumulative history
    "tminus1_2": [
        name_to_idx["current_tminus1_l1"],
        name_to_idx["current_tminus1_l2"],
        name_to_idx["current_tminus2_l1"],
        name_to_idx["current_tminus2_l2"],
    ],

    "tminus1_3": [
        name_to_idx["current_tminus1_l1"],
        name_to_idx["current_tminus1_l2"],
        name_to_idx["current_tminus3_l1"],
        name_to_idx["current_tminus3_l2"],
    ],

    "tminus2_3": [
        name_to_idx["current_tminus2_l1"],
        name_to_idx["current_tminus2_l2"],
        name_to_idx["current_tminus3_l1"],
        name_to_idx["current_tminus3_l2"],
    ],
}

for name, idx in distance_feature_sets.items():
    print(
        f"{name:25s}",
        len(idx),
        [transition_names[i] for i in idx]
    )

zero                      0 []
all_current_distance      6 ['current_tminus3_l1', 'current_tminus3_l2', 'current_tminus2_l1', 'current_tminus2_l2', 'current_tminus1_l1', 'current_tminus1_l2']
l1_only                   3 ['current_tminus3_l1', 'current_tminus2_l1', 'current_tminus1_l1']
l2_only                   3 ['current_tminus3_l2', 'current_tminus2_l2', 'current_tminus1_l2']
tminus1_only              2 ['current_tminus1_l1', 'current_tminus1_l2']
tminus2_only              2 ['current_tminus2_l1', 'current_tminus2_l2']
tminus3_only              2 ['current_tminus3_l1', 'current_tminus3_l2']
tminus1_2                 4 ['current_tminus1_l1', 'current_tminus1_l2', 'current_tminus2_l1', 'current_tminus2_l2']
tminus1_3                 4 ['current_tminus1_l1', 'current_tminus1_l2', 'current_tminus3_l1', 'current_tminus3_l2']
tminus2_3                 4 ['current_tminus2_l1', 'current_tminus2_l2', 'current_tminus3_l1', 'current_tminus3_l2']


In [83]:
# ============================================================
# 67. Cross-fitted models for distance decomposition
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

distance_predictions = {}
distance_probabilities = {}

for experiment, keep_idx in distance_feature_sets.items():

    print(f"\nRunning: {experiment} ({len(keep_idx)} trajectory features)")

    oof_prob = np.zeros(
        (len(y_train), len(CLASS_NAMES)),
        dtype=float,
    )

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X_sem_train,
            y_train,
            groups=groups_train,
        ),
        start=1,
    ):

        # ----------------------------------------------------
        # Build representation
        # ----------------------------------------------------

        if len(keep_idx) > 0:

            X_tr = np.hstack([
                X_sem_train[tr_idx],
                T_train[tr_idx][:, keep_idx],
            ])

            X_va = np.hstack([
                X_sem_train[va_idx],
                T_train[va_idx][:, keep_idx],
            ])

        else:

            X_tr = X_sem_train[tr_idx]
            X_va = X_sem_train[va_idx]

        # ----------------------------------------------------
        # Scale inside fold
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)

        # ----------------------------------------------------
        # Same regularization used in reduced-representation
        # experiment
        # ----------------------------------------------------

        model = LogisticRegression(
            C=0.01,
            max_iter=5000,
            random_state=42,
        )

        model.fit(
            X_tr_scaled,
            y_train[tr_idx],
        )

        oof_prob[va_idx] = model.predict_proba(
            X_va_scaled
        )

    # --------------------------------------------------------
    # Save OOF results
    # --------------------------------------------------------

    pred = oof_prob.argmax(axis=1)

    distance_probabilities[experiment] = oof_prob
    distance_predictions[experiment] = pred

    m = metrics(pred)

    print(
        f"  accuracy={m['accuracy']:.4f} | "
        f"balanced={m['balanced_accuracy']:.4f} | "
        f"macro_f1={m['macro_f1']:.4f}"
    )


print("\nCompleted representations:", list(distance_predictions.keys()))


Running: zero (0 trajectory features)
  accuracy=0.5292 | balanced=0.4777 | macro_f1=0.4908

Running: all_current_distance (6 trajectory features)
  accuracy=0.5379 | balanced=0.4983 | macro_f1=0.5105

Running: l1_only (3 trajectory features)
  accuracy=0.5386 | balanced=0.4979 | macro_f1=0.5102

Running: l2_only (3 trajectory features)
  accuracy=0.5386 | balanced=0.4979 | macro_f1=0.5103

Running: tminus1_only (2 trajectory features)
  accuracy=0.5393 | balanced=0.4936 | macro_f1=0.5070

Running: tminus2_only (2 trajectory features)
  accuracy=0.5306 | balanced=0.4905 | macro_f1=0.5017

Running: tminus3_only (2 trajectory features)
  accuracy=0.5306 | balanced=0.4914 | macro_f1=0.5019

Running: tminus1_2 (4 trajectory features)
  accuracy=0.5393 | balanced=0.4985 | macro_f1=0.5113

Running: tminus1_3 (4 trajectory features)
  accuracy=0.5386 | balanced=0.4998 | macro_f1=0.5118

Running: tminus2_3 (4 trajectory features)
  accuracy=0.5285 | balanced=0.4822 | macro_f1=0.4945

Complete

In [84]:
# ============================================================
# 68. Evaluate distance decomposition
# ============================================================

distance_results = []

zero_pred = distance_predictions["zero"]
zero_correct = zero_pred == y_train

for experiment, pred in distance_predictions.items():

    m = metrics(pred)
    correct = pred == y_train

    rescues = int(
        ((~zero_correct) & correct).sum()
    )

    breaks = int(
        (zero_correct & (~correct)).sum()
    )

    distance_results.append({
        "experiment": experiment,
        "n_features":
            len(distance_feature_sets[experiment]),

        **m,

        "delta_accuracy_vs_zero":
            m["accuracy"] - metrics(zero_pred)["accuracy"],

        "rescues": rescues,
        "breaks": breaks,
        "net": rescues - breaks,

        "changed":
            int((pred != zero_pred).sum()),
    })


distance_results_df = (
    pd.DataFrame(distance_results)
    .sort_values(
        ["macro_f1", "accuracy"],
        ascending=False,
    )
)

display(
    distance_results_df.round(4)
)

,experiment,n_features,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_zero,rescues,breaks,net,changed
8,tminus1_3,4,0.5386,0.4998,0.5118,0.5361,0.0094,32,18,14,62
7,tminus1_2,4,0.5393,0.4985,0.5113,0.5361,0.0101,32,17,15,62
1,all_current_distance,6,0.5379,0.4983,0.5105,0.5352,0.0087,34,21,13,70
3,l2_only,3,0.5386,0.4979,0.5103,0.5356,0.0094,31,17,14,58
2,l1_only,3,0.5386,0.4979,0.5102,0.5356,0.0094,32,18,14,60
4,tminus1_only,2,0.5393,0.4936,0.5070,0.5364,0.0101,29,14,15,53
6,tminus3_only,2,0.5306,0.4914,0.5019,0.5273,0.0013,8,6,2,17
5,tminus2_only,2,0.5306,0.4905,0.5017,0.5268,0.0013,15,13,2,37
9,tminus2_3,4,0.5285,0.4822,0.4945,0.5247,-0.0007,18,19,-1,49
0,zero,0,0.5292,0.4777,0.4908,0.5257,0.0000,0,0,0,0


In [85]:
# ============================================================
# 69. Family recall by distance representation
# ============================================================

family_rows = []

for experiment, pred in distance_predictions.items():

    for class_idx, class_name in enumerate(CLASS_NAMES):

        mask = y_train == class_idx

        family_rows.append({
            "experiment": experiment,
            "failure_family": class_name,
            "support": int(mask.sum()),
            "recall": float(
                (pred[mask] == class_idx).mean()
            ),
        })


distance_family_df = pd.DataFrame(family_rows)

distance_family_pivot = distance_family_df.pivot(
    index="failure_family",
    columns="experiment",
    values="recall",
)

display(
    distance_family_pivot.round(4)
)

experiment,all_current_distance,l1_only,l2_only,tminus1_2,tminus1_3,tminus1_only,tminus2_3,tminus2_only,tminus3_only,zero
failure_family,,,,,,,,,,
constraint_error,0.4448,0.4479,0.4479,0.4511,0.4511,0.4574,0.4259,0.4227,0.4259,0.4290
grounding_state_error,0.4385,0.4344,0.4344,0.4344,0.4426,0.4385,0.4344,0.4385,0.4426,0.4385
reasoning_value_error,0.5161,0.5161,0.5161,0.5161,0.5161,0.4839,0.4839,0.5161,0.5161,0.4516
tool_use_error,0.4346,0.4304,0.4304,0.4304,0.4346,0.4304,0.4093,0.4177,0.4177,0.4135
workflow_error,0.6576,0.6606,0.6606,0.6606,0.6545,0.6576,0.6576,0.6576,0.6545,0.6561


Yes — your interpretation is the right one, and it slightly changes what the next experiment should be.

The current result does not say “t−1 is universally where the failure signal lives.” It says that, under the crude positional representation we built, including the most recent event is important, while t−2/t−3 alone are weak. The strongest compact models are tminus1_2 and tminus1_3, which suggests that recent context plus one comparison point is useful.

But in agent trajectories, “one event ago” is not semantically consistent.

A target ASSISTANT message might have:

t-1 = TOOL_CALL
t-2 = ASSISTANT
t-3 = TOOL_CALL

while another target could have:

t-1 = ASSISTANT
t-2 = TOOL_CALL
t-3 = ASSISTANT

Those positions represent completely different causal relationships.

And your examples are exactly right:

the tool call can be correct, but the following assistant response hallucinates;
the tool call itself can contain the failure through bad arguments or wrong tool selection;
the assistant response can be semantically fine even though the prior action violated a prerequisite;
the relevant relationship may be tool call → next assistant, not “distance to event t−1”;
for another failure it may instead be previous assistant → tool call.

So I would not yet jump directly to generic d1-d2, d2-d3 temporal dynamics. Those are worth testing later, but first we need to make position semantically meaningful.

The next research question should be:

Does trajectory utility depend more on the type of preceding event and transition than on raw chronological lag?

That is stronger and more aligned with your agent setting.

Next experiment: role-aware / transition-aware distance

Build distances such as:

In [86]:
# ============================================================
# 70. Inspect event roles around each target
# ============================================================

from collections import Counter

def recent_role_sequence(
    history_indices,
    events_df,
    k=5,
):
    roles = []

    for indices in history_indices:

        recent = indices[-k:]

        roles.append([
            str(
                events_df.loc[idx, "event_role"]
            )
            for idx in recent
        ])

    return roles


train_recent_roles = (
    recent_role_sequence(
        train_history_indices,
        train_events,
        k=5,
    )
)

role_patterns = Counter(
    tuple(x)
    for x in train_recent_roles
)

print(
    "Unique recent role patterns:",
    len(role_patterns)
)

for pattern, count in (
    role_patterns.most_common(20)
):
    print(
        count,
        pattern,
    )

Unique recent role patterns: 60
325 ('TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL')
71 ('ASSISTANT', 'ASSISTANT', 'ASSISTANT', 'ASSISTANT', 'ASSISTANT')
69 ('ASSISTANT', 'TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL')
61 ('TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL', 'ASSISTANT', 'TOOL_CALL')
56 ()
52 ('TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL', 'ASSISTANT')
50 ('TOOL_CALL', 'ASSISTANT', 'TOOL_CALL', 'ASSISTANT', 'TOOL_CALL')
46 ('TOOL_CALL', 'TOOL_CALL', 'ASSISTANT', 'TOOL_CALL', 'TOOL_CALL')
44 ('TOOL_CALL',)
42 ('TOOL_CALL', 'ASSISTANT', 'TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL')
41 ('TOOL_CALL', 'TOOL_CALL', 'ASSISTANT', 'TOOL_CALL', 'ASSISTANT')
38 ('TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL')
30 ('TOOL_CALL', 'TOOL_CALL')
28 ('ASSISTANT', 'TOOL_CALL', 'ASSISTANT', 'TOOL_CALL', 'ASSISTANT')
28 ('ASSISTANT', 'TOOL_CALL', 'TOOL_CALL', 'TOOL_CALL', 'ASSISTANT')
28 ('TOOL_CALL', 'ASSISTANT', 'TOOL_CALL', 'TOOL_CALL', 'ASSISTANT')
24 ('ASSISTANT', 'TOOL_CALL', 'TOOL_CALL',

In [87]:
# ============================================================
# 71. Role-aware historical embedding lookup
# ============================================================

def build_role_history_embeddings(
    history_indices,
    events_df,
    event_embeddings,
    role,
    max_occurrences=2,
):

    n = len(history_indices)
    d = event_embeddings.shape[1]

    output = np.zeros(
        (
            n,
            max_occurrences,
            d,
        ),
        dtype=np.float32,
    )

    present = np.zeros(
        (
            n,
            max_occurrences,
        ),
        dtype=np.float32,
    )

    for i, indices in enumerate(
        history_indices
    ):

        matching = [
            idx
            for idx in indices
            if str(
                events_df.loc[
                    idx,
                    "event_role",
                ]
            ) == role
        ]

        recent = (
            matching[
                -max_occurrences:
            ][::-1]
        )

        for occurrence, idx in enumerate(
            recent
        ):

            output[
                i,
                occurrence,
            ] = event_embeddings[
                idx
            ]

            present[
                i,
                occurrence,
            ] = 1.0

    return output, present

In [88]:
# ============================================================
# 72. Previous same-role states
# ============================================================

prev_assistant_train, prev_assistant_present = (
    build_role_history_embeddings(
        train_history_indices,
        train_events,
        train_event_embeddings,
        role="ASSISTANT",
        max_occurrences=2,
    )
)

prev_tool_train, prev_tool_present = (
    build_role_history_embeddings(
        train_history_indices,
        train_events,
        train_event_embeddings,
        role="TOOL_CALL",
        max_occurrences=2,
    )
)

print(
    "Assistant:",
    prev_assistant_train.shape
)

print(
    "Tool:",
    prev_tool_train.shape
)

print(
    "\nPrevious assistant available:",
    prev_assistant_present[:, 0].mean()
)

print(
    "Previous tool available:",
    prev_tool_present[:, 0].mean()
)

Assistant: (1489, 2, 384)
Tool: (1489, 2, 384)

Previous assistant available: 0.8629953
Previous tool available: 0.9395568


Assistant: (1489, 2, 384)
Tool: (1489, 2, 384)

Previous assistant available: 0.8629953
Previous tool available: 0.9395568


In [90]:
# ============================================================
# 73. Role-aware distance features
# ============================================================

def row_l1(a, b):
    return np.mean(
        np.abs(a - b),
        axis=1,
    )


def row_l2(a, b):
    return np.linalg.norm(
        a - b,
        axis=1,
    )


role_feature_data = {}

# ------------------------------------------------------------
# Current ↔ latest assistant
# ------------------------------------------------------------

for occurrence in range(2):

    suffix = (
        "prev_assistant"
        if occurrence == 0
        else "prev2_assistant"
    )

    h = prev_assistant_train[
        :,
        occurrence,
        :
    ]

    p = prev_assistant_present[
        :,
        occurrence
    ]

    role_feature_data[
        f"current_{suffix}_l1"
    ] = (
        row_l1(
            X_sem_train,
            h,
        )
        * p
    )

    role_feature_data[
        f"current_{suffix}_l2"
    ] = (
        row_l2(
            X_sem_train,
            h,
        )
        * p
    )

    role_feature_data[
        f"{suffix}_present"
    ] = p


# ------------------------------------------------------------
# Current ↔ latest tool call
# ------------------------------------------------------------

for occurrence in range(2):

    suffix = (
        "prev_tool"
        if occurrence == 0
        else "prev2_tool"
    )

    h = prev_tool_train[
        :,
        occurrence,
        :
    ]

    p = prev_tool_present[
        :,
        occurrence
    ]

    role_feature_data[
        f"current_{suffix}_l1"
    ] = (
        row_l1(
            X_sem_train,
            h,
        )
        * p
    )

    role_feature_data[
        f"current_{suffix}_l2"
    ] = (
        row_l2(
            X_sem_train,
            h,
        )
        * p
    )

    role_feature_data[
        f"{suffix}_present"
    ] = p


role_distance_df = pd.DataFrame(
    role_feature_data
)

print(
    role_distance_df.shape
)

display(
    role_distance_df.head()
)

(1489, 12)


,current_prev_assistant_l1,current_prev_assistant_l2,prev_assistant_present,current_prev2_assistant_l1,current_prev2_assistant_l2,prev2_assistant_present,current_prev_tool_l1,current_prev_tool_l2,prev_tool_present,current_prev2_tool_l1,current_prev2_tool_l2,prev2_tool_present
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.039915,0.973616,1.0,0.040755,0.994857,1.0
1,0.043941,1.090362,1.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.009315,0.230429,1.0,0.010558,0.263075,1.0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.041609,1.002589,1.0,0.041678,1.013256,1.0
4,0.000000,0.000000,0.0,0.0,0.0,0.0,0.032973,0.821834,1.0,0.030622,0.765477,1.0


In [91]:
# ============================================================
# 74. Previous tool ↔ previous assistant relation
# ============================================================

prev_tool_1 = (
    prev_tool_train[:, 0, :]
)

prev_assistant_1 = (
    prev_assistant_train[:, 0, :]
)

pair_present = (
    prev_tool_present[:, 0]
    *
    prev_assistant_present[:, 0]
)

role_distance_df[
    "prev_tool_prev_assistant_l1"
] = (
    row_l1(
        prev_tool_1,
        prev_assistant_1,
    )
    * pair_present
)

role_distance_df[
    "prev_tool_prev_assistant_l2"
] = (
    row_l2(
        prev_tool_1,
        prev_assistant_1,
    )
    * pair_present
)

role_distance_df[
    "tool_assistant_pair_present"
] = pair_present

In [92]:
# ============================================================
# 75. Positional vs role-aware trajectory representation
# ============================================================

positional_idx = (
    distance_feature_sets[
        "all_current_distance"
    ]
)

X_positional = (
    T_train[
        :,
        positional_idx
    ]
)

X_role = (
    role_distance_df
    .to_numpy(
        dtype=np.float32
    )
)

X_positional_role = np.hstack([
    X_positional,
    X_role,
])

print(
    "Positional:",
    X_positional.shape
)

print(
    "Role-aware:",
    X_role.shape
)

print(
    "Combined:",
    X_positional_role.shape
)

Positional: (1489, 6)
Role-aware: (1489, 15)
Combined: (1489, 21)


In [93]:
# ============================================================
# 76. OOF comparison
# ============================================================

trajectory_representations = {
    "semantic_only":
        None,

    "positional_distance":
        X_positional,

    "role_aware_distance":
        X_role,

    "positional_plus_role":
        X_positional_role,
}

role_predictions = {}

for name, X_hist in (
    trajectory_representations.items()
):

    oof_prob = np.zeros(
        (
            len(y_train),
            len(CLASS_NAMES),
        )
    )

    for tr_idx, va_idx in cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ):

        if X_hist is None:

            X_tr = X_sem_train[
                tr_idx
            ]

            X_va = X_sem_train[
                va_idx
            ]

        else:

            X_tr = np.hstack([
                X_sem_train[
                    tr_idx
                ],
                X_hist[
                    tr_idx
                ],
            ])

            X_va = np.hstack([
                X_sem_train[
                    va_idx
                ],
                X_hist[
                    va_idx
                ],
            ])

        scaler = StandardScaler()

        X_tr = scaler.fit_transform(
            X_tr
        )

        X_va = scaler.transform(
            X_va
        )

        model = LogisticRegression(
            C=0.01,
            max_iter=5000,
            random_state=42,
        )

        model.fit(
            X_tr,
            y_train[
                tr_idx
            ],
        )

        oof_prob[
            va_idx
        ] = model.predict_proba(
            X_va
        )

    role_predictions[
        name
    ] = oof_prob.argmax(
        axis=1
    )

In [94]:
# ============================================================
# 77. Role-aware results
# ============================================================

rows = []

semantic_control = (
    role_predictions[
        "semantic_only"
    ]
)

semantic_correct = (
    semantic_control
    == y_train
)

for name, pred in role_predictions.items():

    m = metrics(pred)

    correct = (
        pred == y_train
    )

    rescues = (
        (~semantic_correct)
        & correct
    ).sum()

    breaks = (
        semantic_correct
        & (~correct)
    ).sum()

    rows.append({
        "model": name,
        **m,
        "rescues": rescues,
        "breaks": breaks,
        "net":
            rescues - breaks,
    })

role_results_df = (
    pd.DataFrame(rows)
    .sort_values(
        "macro_f1",
        ascending=False,
    )
)

display(
    role_results_df.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,rescues,breaks,net
1,positional_distance,0.5379,0.4983,0.5105,0.5352,34,21,13
3,positional_plus_role,0.5386,0.4852,0.5000,0.5350,57,43,14
2,role_aware_distance,0.5359,0.4843,0.4989,0.5324,46,36,10
0,semantic_only,0.5292,0.4777,0.4908,0.5257,0,0,0


These old MoE results fit the current notebook extremely well. In fact, they strengthen the direction we’ve now converged on.

Your earlier MoE work already showed that the semantic and structured experts are complementary, but that routing is the bottleneck rather than expert diversity itself. The structured expert was much weaker globally, yet the oracle reached about 0.617 accuracy vs 0.516 for semantic, proving there was substantial complementary signal available. Learned routing only produced small gains, and the direct routers did not beat an always-semantic selection policy on the subset where routing mattered. The weighted rescue router looked promising OOF but actually reversed on the held-out test set, going from +28 net rescues OOF to −5 on test, which is exactly the kind of overfitting we want to avoid repeating.

That old notebook ended with the right research decision: stop handcrafted routing and investigate the representation problem instead. Notebook 15 is now giving us evidence for what that representation problem actually is.

What the new role-aware result says

You have 60 distinct recent role patterns. Even among the 20 most common, the sequences vary dramatically:

TOOL TOOL TOOL TOOL TOOL
ASSISTANT ASSISTANT ASSISTANT ASSISTANT ASSISTANT
ASSISTANT TOOL TOOL TOOL TOOL
TOOL ASSISTANT TOOL ASSISTANT TOOL
...

So t−1, t−2, and t−3 are clearly not stable semantic coordinates.

That validates your intuition: a failure can live in different event locations depending on what happened.

Yet the first role-aware representation did not beat positional distance:

Representation	Accuracy	Balanced Acc.	Macro-F1	Net
Positional distance	0.5379	0.4983	0.5105	+13
Role-aware distance	0.5359	0.4843	0.4989	+10
Positional + role	0.5386	0.4852	0.5000	+14
Semantic control	0.5292	0.4777	0.4908	—

So simply replacing chronological position with:

previous TOOL_CALL
previous ASSISTANT
second previous TOOL_CALL
second previous ASSISTANT

doesn't solve it.

That doesn't mean event roles are unimportant. It means role identity alone is still too coarse.

A TOOL_CALL can be:

wrong tool,
wrong argument,
missing required argument,
correct call after a missing prerequisite,
correct call followed by a bad response,
repeated tool invocation,
or a completely valid intermediate action.

Likewise an ASSISTANT event can contain grounding failure, workflow failure, constraint failure, or no failure at all.

So what we really care about is not:

"What role occurred?"

but:

"What relation does the current failure-bearing event have to the preceding action/state?"

That connects directly back to the old MoE conclusion. The structured branch had useful complementary signal, but its representation did not expose that information in a stable enough way for routing.

I would change the next experiment slightly

I would not continue expanding role-aware embeddings, and I would also not return to MoE routing yet.

Your earlier proposed temporal-dynamics experiment is now worth doing, but we should make it event-structure aware.

We already know:

t−1 matters;
t−1 + t−2 and t−1 + t−3 outperform either older lag alone;
raw role-aware distances don't improve macro-F1;
therefore older events may matter mainly by telling us how the relationship to the current event is changing.

So now test:

Is the useful feature the distance itself, or the change in distance across preceding events?

Next experiment — temporal distance dynamics

In [95]:
# ============================================================
# 78. Explicit temporal distance dynamics
# ============================================================

eps = 1e-8

t1_l1 = T_train[
    :,
    name_to_idx["current_tminus1_l1"]
]

t2_l1 = T_train[
    :,
    name_to_idx["current_tminus2_l1"]
]

t3_l1 = T_train[
    :,
    name_to_idx["current_tminus3_l1"]
]


t1_l2 = T_train[
    :,
    name_to_idx["current_tminus1_l2"]
]

t2_l2 = T_train[
    :,
    name_to_idx["current_tminus2_l2"]
]

t3_l2 = T_train[
    :,
    name_to_idx["current_tminus3_l2"]
]


p1 = T_train[
    :,
    name_to_idx["tminus1_present"]
]

p2 = T_train[
    :,
    name_to_idx["tminus2_present"]
]

p3 = T_train[
    :,
    name_to_idx["tminus3_present"]
]

In [96]:
# ============================================================
# 79. Dynamic trajectory features
# ============================================================

dynamic_features = pd.DataFrame({
    # --------------------------------------------
    # Absolute recent distance
    # --------------------------------------------
    "t1_l1": t1_l1,
    "t1_l2": t1_l2,

    # --------------------------------------------
    # Change in distance
    # positive => current is farther from recent
    # event than older event
    # --------------------------------------------
    "l1_t1_minus_t2":
        (t1_l1 - t2_l1)
        * p1 * p2,

    "l1_t2_minus_t3":
        (t2_l1 - t3_l1)
        * p2 * p3,

    "l1_t1_minus_t3":
        (t1_l1 - t3_l1)
        * p1 * p3,

    "l2_t1_minus_t2":
        (t1_l2 - t2_l2)
        * p1 * p2,

    "l2_t2_minus_t3":
        (t2_l2 - t3_l2)
        * p2 * p3,

    "l2_t1_minus_t3":
        (t1_l2 - t3_l2)
        * p1 * p3,

    # --------------------------------------------
    # Relative change
    # --------------------------------------------
    "l1_t1_over_t2":
        np.where(
            (p1 * p2) > 0,
            t1_l1 / (t2_l1 + eps),
            0.0,
        ),

    "l1_t1_over_t3":
        np.where(
            (p1 * p3) > 0,
            t1_l1 / (t3_l1 + eps),
            0.0,
        ),

    "l2_t1_over_t2":
        np.where(
            (p1 * p2) > 0,
            t1_l2 / (t2_l2 + eps),
            0.0,
        ),

    "l2_t1_over_t3":
        np.where(
            (p1 * p3) > 0,
            t1_l2 / (t3_l2 + eps),
            0.0,
        ),

    # --------------------------------------------
    # Availability
    # --------------------------------------------
    "t1_present": p1,
    "t2_present": p2,
    "t3_present": p3,
})

print(
    dynamic_features.shape
)

display(
    dynamic_features.head()
)

(1489, 15)


,t1_l1,t1_l2,l1_t1_minus_t2,l1_t2_minus_t3,l1_t1_minus_t3,l2_t1_minus_t2,l2_t2_minus_t3,l2_t1_minus_t3,l1_t1_over_t2,l1_t1_over_t3,l2_t1_over_t2,l2_t1_over_t3,t1_present,t2_present,t3_present
0,0.039915,0.973616,-0.000840,0.000000,0.000000,-0.021242,0.000000,0.00000,0.979397,0.000000,0.978648,0.000000,1.0,1.0,0.0
1,0.043941,1.090362,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.0,0.0,0.0
2,0.009315,0.230429,-0.001243,0.000000,0.000000,-0.032646,0.000000,0.00000,0.882248,0.000000,0.875907,0.000000,1.0,1.0,0.0
3,0.041609,1.002589,-0.000069,-0.000964,-0.001033,-0.010666,-0.030623,-0.04129,0.998346,0.975781,0.989473,0.960446,1.0,1.0,1.0
4,0.032973,0.821834,0.002351,-0.002485,-0.000134,0.056357,-0.059617,-0.00326,1.076772,0.995965,1.073623,0.996049,1.0,1.0,1.0


In [97]:
# ============================================================
# 80. Dynamic feature hypotheses
# ============================================================

dynamic_feature_sets = {
    "semantic_only": [],

    # current best local state
    "recent_distance": [
        "t1_l1",
        "t1_l2",
    ],

    # only change information
    "difference_only": [
        "l1_t1_minus_t2",
        "l1_t2_minus_t3",
        "l1_t1_minus_t3",
        "l2_t1_minus_t2",
        "l2_t2_minus_t3",
        "l2_t1_minus_t3",
    ],

    "ratio_only": [
        "l1_t1_over_t2",
        "l1_t1_over_t3",
        "l2_t1_over_t2",
        "l2_t1_over_t3",
    ],

    # key hypotheses
    "recent_plus_difference": [
        "t1_l1",
        "t1_l2",
        "l1_t1_minus_t2",
        "l1_t1_minus_t3",
        "l2_t1_minus_t2",
        "l2_t1_minus_t3",
    ],

    "recent_plus_ratio": [
        "t1_l1",
        "t1_l2",
        "l1_t1_over_t2",
        "l1_t1_over_t3",
        "l2_t1_over_t2",
        "l2_t1_over_t3",
    ],

    "all_dynamic": list(
        dynamic_features.columns
    ),
}

In [98]:
# ============================================================
# 81. Cross-fitted dynamic representation evaluation
# ============================================================

dynamic_predictions = {}

for experiment, cols in dynamic_feature_sets.items():

    print("\n", experiment)

    oof_prob = np.zeros(
        (
            len(y_train),
            len(CLASS_NAMES),
        ),
        dtype=float,
    )

    if len(cols) > 0:
        X_dynamic = (
            dynamic_features[
                cols
            ]
            .to_numpy(
                dtype=np.float32
            )
        )
    else:
        X_dynamic = None


    for tr_idx, va_idx in cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ):

        if X_dynamic is None:

            X_tr = X_sem_train[
                tr_idx
            ]

            X_va = X_sem_train[
                va_idx
            ]

        else:

            X_tr = np.hstack([
                X_sem_train[
                    tr_idx
                ],
                X_dynamic[
                    tr_idx
                ],
            ])

            X_va = np.hstack([
                X_sem_train[
                    va_idx
                ],
                X_dynamic[
                    va_idx
                ],
            ])

        scaler = StandardScaler()

        X_tr_scaled = (
            scaler.fit_transform(
                X_tr
            )
        )

        X_va_scaled = (
            scaler.transform(
                X_va
            )
        )

        model = LogisticRegression(
            C=0.01,
            max_iter=5000,
            random_state=42,
        )

        model.fit(
            X_tr_scaled,
            y_train[
                tr_idx
            ],
        )

        oof_prob[
            va_idx
        ] = model.predict_proba(
            X_va_scaled
        )


    dynamic_predictions[
        experiment
    ] = oof_prob.argmax(
        axis=1
    )


 semantic_only

 recent_distance

 difference_only

 ratio_only

 recent_plus_difference

 recent_plus_ratio

 all_dynamic


In [99]:
# ============================================================
# 82. Compare dynamic trajectory representation
# ============================================================

rows = []

semantic_pred = dynamic_predictions[
    "semantic_only"
]

semantic_correct = (
    semantic_pred == y_train
)


for experiment, pred in (
    dynamic_predictions.items()
):

    m = metrics(pred)

    correct = (
        pred == y_train
    )

    rescues = int(
        (
            (~semantic_correct)
            & correct
        ).sum()
    )

    breaks = int(
        (
            semantic_correct
            & (~correct)
        ).sum()
    )

    rows.append({
        "experiment":
            experiment,

        "n_dynamic_features":
            len(
                dynamic_feature_sets[
                    experiment
                ]
            ),

        **m,

        "rescues":
            rescues,

        "breaks":
            breaks,

        "net":
            rescues - breaks,
    })


dynamic_results_df = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "macro_f1",
            "accuracy",
        ],
        ascending=False,
    )
)

display(
    dynamic_results_df.round(4)
)

,experiment,n_dynamic_features,accuracy,balanced_accuracy,macro_f1,weighted_f1,rescues,breaks,net
5,recent_plus_ratio,6,0.5400,0.4947,0.5081,0.5373,31,15,16
1,recent_distance,2,0.5393,0.4936,0.5070,0.5364,29,14,15
4,recent_plus_difference,6,0.5386,0.4917,0.5055,0.5355,36,22,14
6,all_dynamic,15,0.5373,0.4857,0.4995,0.5348,44,32,12
2,difference_only,6,0.5319,0.4806,0.4934,0.5287,16,12,4
3,ratio_only,4,0.5306,0.4803,0.4931,0.5273,4,2,2
0,semantic_only,0,0.5292,0.4777,0.4908,0.5257,0,0,0


In [100]:
print(
    "\nReference:"
)

print(
    "tminus1_3:"
    " accuracy=0.5386,"
    " balanced=0.4998,"
    " macro_f1=0.5118"
)

print(
    "tminus1_2:"
    " accuracy=0.5393,"
    " balanced=0.4985,"
    " macro_f1=0.5113"
)


Reference:
tminus1_3: accuracy=0.5386, balanced=0.4998, macro_f1=0.5118
tminus1_2: accuracy=0.5393, balanced=0.4985, macro_f1=0.5113


Yes — and the zero-history cases are important enough that I would explicitly separate them from the temporal-dynamics conclusion.

The new dynamic experiment is useful, but it does not beat the simpler positional-distance representation on the class-balanced objective:

Representation	Accuracy	Balanced Acc.	Macro-F1
tminus1_3	0.5386	0.4998	0.5118
tminus1_2	0.5393	0.4985	0.5113
recent_plus_ratio	0.5400	0.4947	0.5081
recent_distance	0.5393	0.4936	0.5070
recent_plus_difference	0.5386	0.4917	0.5055
semantic only	0.5292	0.4777	0.4908

So ratios add a tiny amount of raw accuracy, but reduce balanced accuracy and macro-F1 relative to the simple t−1 + t−3 representation. Differences are even less convincing. This tells us that the older event is useful, but not primarily because the model needs a hand-engineered velocity/acceleration feature.

The cleaner interpretation is:

Recent distance contains the main trajectory signal, while an additional older reference point contributes complementary context. Explicit temporal derivatives do not improve the representation beyond retaining the underlying distances themselves.

That is useful because it argues against continuing to manufacture increasingly elaborate temporal features.

The zero-history issue sharpens this further. You have 56 zero-history examples. For them, all trajectory features are exactly zero, so no notion of t−1, ratio, difference, or role transition exists. They should really be viewed as a separate regime:

history unavailable
    → semantic-only information


history available
    → semantic + trajectory-distance information

That means our reported trajectory improvements are currently diluted by 56 examples for which trajectory can never contribute. The next experiment should quantify trajectory utility conditional on history availability, and then by actual available depth.

In [101]:
# ============================================================
# 83. History-availability regimes
# ============================================================

history_count = (
    train_targets[
        "history_event_count"
    ]
    .to_numpy()
    .astype(int)
)

history_regime = np.select(
    [
        history_count == 0,
        history_count == 1,
        history_count == 2,
        history_count >= 3,
    ],
    [
        "zero_history",
        "one_event",
        "two_events",
        "three_plus",
    ],
    default="unknown",
)

history_regime_df = pd.DataFrame({
    "history_event_count": history_count,
    "history_regime": history_regime,
    "target": y_train,
})

print(
    pd.Series(history_regime)
    .value_counts()
)

three_plus      1313
two_events        63
one_event         57
zero_history      56
Name: count, dtype: int64


In [102]:
# ============================================================
# 84. Performance conditioned on available history
# ============================================================

models_for_depth = {
    "semantic_control":
        distance_predictions["zero"],

    "tminus1_only":
        distance_predictions["tminus1_only"],

    "tminus1_2":
        distance_predictions["tminus1_2"],

    "tminus1_3":
        distance_predictions["tminus1_3"],

    "all_current_distance":
        distance_predictions["all_current_distance"],
}

depth_rows = []

for regime in [
    "zero_history",
    "one_event",
    "two_events",
    "three_plus",
]:

    mask = (
        history_regime == regime
    )

    for model_name, pred in models_for_depth.items():

        if mask.sum() == 0:
            continue

        depth_rows.append({
            "history_regime":
                regime,

            "model":
                model_name,

            "support":
                int(mask.sum()),

            "accuracy":
                accuracy_score(
                    y_train[mask],
                    pred[mask],
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_train[mask],
                    pred[mask],
                ),

            "macro_f1":
                f1_score(
                    y_train[mask],
                    pred[mask],
                    average="macro",
                    zero_division=0,
                ),
        })


depth_results_df = pd.DataFrame(
    depth_rows
)

display(
    depth_results_df
    .sort_values(
        [
            "history_regime",
            "macro_f1",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(4)
)

,history_regime,model,support,accuracy,balanced_accuracy,macro_f1
6,one_event,tminus1_only,57,0.5614,0.5980,0.5256
8,one_event,tminus1_3,57,0.5614,0.5980,0.5254
7,one_event,tminus1_2,57,0.5439,0.5896,0.5184
5,one_event,semantic_control,57,0.5263,0.5770,0.5062
9,one_event,all_current_distance,57,0.5614,0.5912,0.5038
17,three_plus,tminus1_2,1313,0.5446,0.4974,0.5134
18,three_plus,tminus1_3,1313,0.5423,0.4964,0.5119
19,three_plus,all_current_distance,1313,0.5423,0.4953,0.5112
16,three_plus,tminus1_only,1313,0.5430,0.4894,0.5063
15,three_plus,semantic_control,1313,0.5354,0.4736,0.4902


In [103]:
# ============================================================
# 85. Incremental utility by history depth
# ============================================================

baseline = (
    distance_predictions["zero"]
)

baseline_correct = (
    baseline == y_train
)

increment_rows = []

comparisons = {
    "tminus1_only":
        distance_predictions["tminus1_only"],

    "tminus1_2":
        distance_predictions["tminus1_2"],

    "tminus1_3":
        distance_predictions["tminus1_3"],

    "all_current_distance":
        distance_predictions["all_current_distance"],
}


for model_name, pred in comparisons.items():

    correct = (
        pred == y_train
    )

    for regime in [
        "zero_history",
        "one_event",
        "two_events",
        "three_plus",
    ]:

        mask = (
            history_regime == regime
        )

        rescue = (
            mask
            & (~baseline_correct)
            & correct
        )

        brk = (
            mask
            & baseline_correct
            & (~correct)
        )

        increment_rows.append({
            "model":
                model_name,

            "history_regime":
                regime,

            "support":
                int(mask.sum()),

            "rescues":
                int(rescue.sum()),

            "breaks":
                int(brk.sum()),

            "net":
                int(
                    rescue.sum()
                    - brk.sum()
                ),

            "net_rate":
                (
                    rescue.sum()
                    - brk.sum()
                ) / mask.sum()
                if mask.sum() > 0
                else np.nan,
        })


history_increment_df = pd.DataFrame(
    increment_rows
)

display(
    history_increment_df
    .sort_values(
        [
            "history_regime",
            "net_rate",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(4)
)

,model,history_regime,support,rescues,breaks,net,net_rate
1,tminus1_only,one_event,57,2,0,2,0.0351
9,tminus1_3,one_event,57,2,0,2,0.0351
13,all_current_distance,one_event,57,3,1,2,0.0351
5,tminus1_2,one_event,57,2,1,1,0.0175
7,tminus1_2,three_plus,1313,26,14,12,0.0091
3,tminus1_only,three_plus,1313,23,13,10,0.0076
11,tminus1_3,three_plus,1313,25,16,9,0.0069
15,all_current_distance,three_plus,1313,26,17,9,0.0069
2,tminus1_only,two_events,63,0,0,0,0.0000
10,tminus1_3,two_events,63,1,1,0,0.0000


In [104]:
# ============================================================
# 86. Does each additional historical event add value?
# ============================================================

nested_rows = []

nested_tests = [
    (
        "t1_vs_semantic",
        distance_predictions["zero"],
        distance_predictions["tminus1_only"],
        history_count >= 1,
    ),
    (
        "t1_2_vs_t1",
        distance_predictions["tminus1_only"],
        distance_predictions["tminus1_2"],
        history_count >= 2,
    ),
    (
        "t1_3_vs_t1",
        distance_predictions["tminus1_only"],
        distance_predictions["tminus1_3"],
        history_count >= 3,
    ),
    (
        "t1_3_vs_t1_2",
        distance_predictions["tminus1_2"],
        distance_predictions["tminus1_3"],
        history_count >= 3,
    ),
]


for (
    name,
    base_pred,
    new_pred,
    eligible,
) in nested_tests:

    base_correct = (
        base_pred == y_train
    )

    new_correct = (
        new_pred == y_train
    )

    rescues = (
        eligible
        & (~base_correct)
        & new_correct
    )

    breaks = (
        eligible
        & base_correct
        & (~new_correct)
    )

    nested_rows.append({
        "comparison":
            name,

        "eligible_n":
            int(eligible.sum()),

        "base_accuracy":
            accuracy_score(
                y_train[eligible],
                base_pred[eligible],
            ),

        "new_accuracy":
            accuracy_score(
                y_train[eligible],
                new_pred[eligible],
            ),

        "rescues":
            int(rescues.sum()),

        "breaks":
            int(breaks.sum()),

        "net":
            int(
                rescues.sum()
                - breaks.sum()
            ),
    })


nested_history_df = pd.DataFrame(
    nested_rows
)

display(
    nested_history_df.round(4)
)

,comparison,eligible_n,base_accuracy,new_accuracy,rescues,breaks,net
0,t1_vs_semantic,1433,0.5304,0.5387,25,13,12
1,t1_2_vs_t1,1376,0.5378,0.5385,8,7,1
2,t1_3_vs_t1,1313,0.5430,0.5423,5,6,-1
3,t1_3_vs_t1_2,1313,0.5446,0.5423,8,11,-3


This gives us a much cleaner conclusion, and it also exposes why the zero-history rows must be handled carefully.

The strongest result is the nested eligible-history comparison, not the raw regime table. When history actually exists, adding the most recent event gives a meaningful gain:

t1 vs semantic, eligible n=1433: accuracy 0.5304 → 0.5387, with 25 rescues / 13 breaks = +12 net.
Adding t−2 after t−1 contributes almost nothing: +1 net across 1,376 eligible examples.
Adding t−3 after t−1 is slightly negative: −1 net.
Replacing t1+t2 with t1+t3 among examples with ≥3 events is worse: −3 net.

So the strongest evidence now is:

Most usable trajectory signal comes from the immediately preceding event. Older history provides little additional predictive value once t−1 is available.

The three_plus regime supports the same interpretation. tminus1_only already raises accuracy from 0.5354 → 0.5430. tminus1_2 reaches 0.5446, only a tiny additional gain, while tminus1_3 falls to 0.5423.

The zero-history rows are not trajectory evidence

Your zero-history table shows trajectory models outperforming the semantic control even though every history feature is zero:

semantic control: 0.5000 accuracy
distance models: 0.5536

That cannot be attributed to historical information.

This happens because these are separately retrained classifiers. Adding trajectory columns during training changes the fitted semantic coefficients and regularized decision boundary, even when a particular evaluation row has all-zero trajectory features.

So this:

zero-history:
semantic control    0.5000
tminus1 model       0.5536

is a model-fitting effect, not a trajectory effect.

That is the same methodological issue we uncovered earlier with the apparent constraint rescues.

For the final notebook conclusions, I would explicitly say:

Zero-history examples were retained in overall evaluation, but differences between separately trained trajectory and semantic models on those examples cannot be interpreted as historical-context effects because no historical information is present.

We should now do one final clean lag-ablation experiment

We have enough evidence to stop expanding the feature space. But before declaring “t−1 is the useful historical signal,” let's isolate the contribution of t−1/t−2/t−3 inside the same fitted six-distance-feature model.

This is the clean counterpart to the retrained representation experiments.

Train the six-distance model normally. At inference time:

full history
remove t−1 only
remove t−2 only
remove t−3 only
zero all history

Same coefficients, same scaler, same semantic embedding. Only historical evidence changes.

In [105]:
# ============================================================
# 87. Clean lag-specific intervention
#
# Same fitted six-distance model.
# Remove individual historical lags only at inference time.
# ============================================================

distance_names = [
    "current_tminus3_l1",
    "current_tminus3_l2",
    "current_tminus2_l1",
    "current_tminus2_l2",
    "current_tminus1_l1",
    "current_tminus1_l2",
]

distance_idx = [
    name_to_idx[name]
    for name in distance_names
]

# positions within the reduced 6-feature block
lag_positions = {
    "tminus3": [0, 1],
    "tminus2": [2, 3],
    "tminus1": [4, 5],
}

interventions = {
    "full_distance": [],
    "remove_tminus1": lag_positions["tminus1"],
    "remove_tminus2": lag_positions["tminus2"],
    "remove_tminus3": lag_positions["tminus3"],
    "keep_tminus1_only": (
        lag_positions["tminus2"]
        + lag_positions["tminus3"]
    ),
    "zero_history": list(range(6)),
}

lag_intervention_prob = {
    name: np.zeros(
        (len(y_train), len(CLASS_NAMES)),
        dtype=float,
    )
    for name in interventions
}


for fold, (tr_idx, va_idx) in enumerate(
    cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    # --------------------------------------------------------
    # Train ONCE using all six distance features
    # --------------------------------------------------------

    T_tr_distance = T_train[
        tr_idx
    ][:, distance_idx]

    T_va_distance = T_train[
        va_idx
    ][:, distance_idx]

    X_tr = np.hstack([
        X_sem_train[tr_idx],
        T_tr_distance,
    ])

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(
        X_tr
    )

    model = LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_tr_scaled,
        y_train[tr_idx],
    )

    # --------------------------------------------------------
    # Same model — different history intervention
    # --------------------------------------------------------

    for intervention, remove_positions in interventions.items():

        T_counterfactual = (
            T_va_distance.copy()
        )

        if remove_positions:
            T_counterfactual[
                :,
                remove_positions
            ] = 0.0

        X_va = np.hstack([
            X_sem_train[va_idx],
            T_counterfactual,
        ])

        lag_intervention_prob[
            intervention
        ][va_idx] = model.predict_proba(
            scaler.transform(X_va)
        )

    print(
        f"Fold {fold} complete"
    )


lag_intervention_pred = {
    name: prob.argmax(axis=1)
    for name, prob in lag_intervention_prob.items()
}

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [106]:
# ============================================================
# 88. Lag intervention results
# ============================================================

full_pred = lag_intervention_pred[
    "full_distance"
]

full_correct = (
    full_pred == y_train
)

rows = []

for intervention, pred in (
    lag_intervention_pred.items()
):

    m = metrics(pred)

    correct = (
        pred == y_train
    )

    # Relative to FULL trajectory model:
    #
    # loss_when_removed =
    # full was correct, intervention became wrong
    #
    # gain_when_removed =
    # full was wrong, intervention became correct

    lost = (
        full_correct
        &
        (~correct)
    )

    gained = (
        (~full_correct)
        &
        correct
    )

    rows.append({
        "intervention":
            intervention,

        **m,

        "changed_vs_full":
            int(
                (
                    pred != full_pred
                ).sum()
            ),

        "gained_when_removed":
            int(gained.sum()),

        "lost_when_removed":
            int(lost.sum()),

        "net_effect_of_removal":
            int(
                gained.sum()
                - lost.sum()
            ),
    })


lag_intervention_results = (
    pd.DataFrame(rows)
    .sort_values(
        "macro_f1",
        ascending=False,
    )
)

display(
    lag_intervention_results.round(4)
)

,intervention,accuracy,balanced_accuracy,macro_f1,weighted_f1,changed_vs_full,gained_when_removed,lost_when_removed,net_effect_of_removal
0,full_distance,0.5379,0.4983,0.5105,0.5352,0,0,0,0
4,keep_tminus1_only,0.5353,0.4879,0.5033,0.5302,75,25,29,-4
3,remove_tminus3,0.5319,0.4907,0.5004,0.5319,69,22,31,-9
2,remove_tminus2,0.5393,0.4816,0.4987,0.5303,91,35,33,2
5,zero_history,0.5212,0.4340,0.4585,0.5017,217,68,93,-25
1,remove_tminus1,0.5151,0.4355,0.4548,0.4969,181,53,87,-34


In [107]:
# ============================================================
# 89. Lag contribution on eligible examples only
# ============================================================

lag_eligibility = {
    "tminus1": history_count >= 1,
    "tminus2": history_count >= 2,
    "tminus3": history_count >= 3,
}

rows = []

for lag in [
    "tminus1",
    "tminus2",
    "tminus3",
]:

    intervention = (
        f"remove_{lag}"
    )

    mask = lag_eligibility[
        lag
    ]

    removed_pred = (
        lag_intervention_pred[
            intervention
        ]
    )

    removed_correct = (
        removed_pred == y_train
    )

    gained = (
        mask
        &
        (~full_correct)
        &
        removed_correct
    )

    lost = (
        mask
        &
        full_correct
        &
        (~removed_correct)
    )

    rows.append({
        "lag":
            lag,

        "eligible_n":
            int(mask.sum()),

        "full_accuracy":
            accuracy_score(
                y_train[mask],
                full_pred[mask],
            ),

        "without_lag_accuracy":
            accuracy_score(
                y_train[mask],
                removed_pred[mask],
            ),

        "gained_when_removed":
            int(gained.sum()),

        "lost_when_removed":
            int(lost.sum()),

        "net_removal":
            int(
                gained.sum()
                - lost.sum()
            ),
    })


eligible_lag_effect_df = (
    pd.DataFrame(rows)
)

display(
    eligible_lag_effect_df.round(4)
)

,lag,eligible_n,full_accuracy,without_lag_accuracy,gained_when_removed,lost_when_removed,net_removal
0,tminus1,1433,0.5373,0.5136,53,87,-34
1,tminus2,1376,0.5363,0.5378,35,33,2
2,tminus3,1313,0.5423,0.5354,22,31,-9


Yes. This is the clean intervention result we needed, and it substantially sharpens the trajectory hypothesis.

### What the intervention establishes

Because the **same fitted six-distance model** is used and only particular lag features are zeroed at inference time, these effects are much more interpretable than the earlier separately retrained ablations.

The result for `t−1` is decisive:

* Full model accuracy: **0.5379**
* Remove `t−1`: **0.5151**
* On the 1,433 eligible examples: **0.5373 → 0.5136**
* 53 examples improve when `t−1` is removed, but **87 deteriorate**
* Net removal effect: **−34**

So `t−1` is not merely correlated with a better retrained model. **The fitted transition classifier is actively using the immediately preceding event to make useful decisions.**

That confirms the central hypothesis.

`t−2` behaves very differently. Removing it gives **35 gains versus 33 losses**, net **+2**, and eligible accuracy actually changes from 0.5363 to **0.5378**. Conditional on the other trajectory information, `t−2` has essentially **zero net utility**.

`t−3` is more interesting. Removing it produces 22 gains versus 31 losses, net **−9**, and eligible accuracy falls from **0.5423 → 0.5354**. Thus `t−3` contains some real complementary signal. It is weaker than `t−1`, but it is not simply noise.

The ordering is therefore:

**t−1 ≫ t−3 > t−2**

This also explains the earlier representation experiment remarkably well. `tminus1_3` had the best balanced accuracy/macro-F1 (**0.4998 / 0.5118**), whereas `tminus2_3` was essentially useless and slightly negative. The intervention now tells us *why*: t−1 carries the dominant information, t−3 contributes a smaller complementary component, while t−2 contributes almost nothing conditional on them.

### One subtle but important result

`keep_tminus1_only` is not equivalent to saying older history is useless.

Removing both t−2 and t−3 simultaneously gives:

* accuracy 0.5353
* 25 gains / 29 losses
* net −4

Yet removing t−3 alone gives net −9 and removing t−2 alone gives +2.

That suggests **redundancy/interactions among older lags**. In particular, some information carried by t−3 may become unnecessary or differently interpreted when t−2 is also absent. We should therefore avoid claiming that lag contributions are additive.

The defensible conclusion is:

> The immediately preceding event provides the dominant trajectory signal. The second preceding event has negligible conditional utility, while the third preceding event provides a smaller complementary signal whose value depends on the surrounding lag representation.

### Zeroing all trajectory information confirms aggregate utility

The full-to-zero intervention is also useful:

**0.5379 → 0.5212**, with 68 gains but **93 losses**, net **−25**.

So the six distance features have positive aggregate utility. More importantly, removing `t−1` alone causes **−34 net**, even larger than removing all history.

That apparent paradox is informative rather than contradictory: removing everything also removes some **harmful/noisy older-history effects**, partially compensating for the loss of t−1.

This is another argument against simply feeding all available history to the classifier.

---

## What this means for the research direction

I would now stop investigating arbitrary `t−k` positional features.

We have enough evidence from several independent experiments:

1. Six current-distance features outperform semantic control.
2. `t−1` alone captures most of the useful improvement.
3. Adding `t−2` gives essentially no incremental benefit.
4. `t−3` has weaker but real complementary value.
5. Explicit difference/ratio dynamics don't improve substantially over raw recent distance.
6. Role-aware history did not outperform positional distance.
7. The clean same-model intervention directly verifies that removing t−1 is strongly harmful.
8. All-history representations can actually be worse than compact subsets.

That points toward a more interesting interpretation:

> **Useful trajectory information is sparse and event-local, rather than a generic summary of historical trajectory.**

And that connects directly to your earlier observation about agent failures.

`t−1` does not necessarily mean the same *semantic relationship* across examples. It could be:

```text
TOOL_CALL → current ASSISTANT
ASSISTANT → current TOOL_CALL
TOOL_RESULT → current ASSISTANT
TOOL_CALL → current TOOL_CALL
ASSISTANT → current ASSISTANT
```

Distance to the previous event is useful, but we're currently asking one numerical feature to represent all of these fundamentally different transitions.

That is probably the next bottleneck.

## Next experiment: condition t−1 distance on transition type

Rather than another distance transformation, I would test:

> **Does the utility of recent displacement depend on the role/event transition that produced it?**

For each example, construct something like:

```text
previous_event_role
current_event_role

transition_type =
    previous_event_role + "->" + current_event_role
```

Then evaluate the already-established `t−1` intervention separately by transition type.

Crucially, **don't train another model yet**. First diagnose where t−1 is helping.

We want a table conceptually like:

| transition            | support | full acc | no-t1 acc | gains removing t1 | losses removing t1 | net t1 utility |
| --------------------- | ------: | -------: | --------: | ----------------: | -----------------: | -------------: |
| TOOL_CALL → ASSISTANT |     ... |      ... |       ... |               ... |                ... |            ... |
| ASSISTANT → TOOL_CALL |     ... |      ... |       ... |               ... |                ... |            ... |
| TOOL_CALL → TOOL_CALL |     ... |      ... |       ... |               ... |                ... |            ... |
| ASSISTANT → ASSISTANT |     ... |      ... |       ... |               ... |                ... |            ... |

And then cross it with the **true failure family**:

```text
transition_type × failure_family × t1 intervention effect
```

That directly tests your example:

> Is recent displacement useful because an assistant response follows a tool action? Is it useful when detecting bad tool arguments? Does it hurt constraint classification because the relevant event relationship is different?

This is much more informative now than engineering more `d1-d2`, ratios, cosines, or arbitrary lag combinations.

### Current notebook conclusion to preserve

I would record this as the key result:

> **Lag-specific intervention demonstrates that trajectory utility is concentrated in recent context. Removing t−1 from a fixed full-distance classifier reduces accuracy from 0.5379 to 0.5151 and produces a net 34 prediction failures among eligible examples. Removing t−2 is neutral (+2 net), while removing t−3 is moderately harmful (−9 net). Thus historical utility is strongly non-uniform across temporal positions: the immediately preceding event is the dominant source of useful trajectory information, with weaker complementary information at t−3 and negligible conditional contribution from t−2. The non-additivity of lag ablations further indicates interactions/redundancy among historical features. These findings motivate replacing generic positional-history modeling with event-relationship-aware analysis, beginning with the interaction between previous/current event roles and t−1 displacement.**

**Next notebook cell should therefore be diagnostic transition-type × t−1 utility analysis, not another classifier.**


# Notebook 15 — `15_constraint_error_trajectory_mechanisms.ipynb`

## Research objective

Notebook 15 moved beyond the question **“does trajectory context help?”** and investigated the more important mechanistic question:

> **What information in trajectory context actually causes useful prediction changes, where does it help, where does it hurt, and what representation should a reliable agent-failure classifier use?**

The notebook was motivated by the previous trajectory/context work, where transition features showed measurable discriminative signal but learned routing/selectors failed to translate that signal into robust classification gains.

The central finding of Notebook 15 is that **trajectory information is genuinely useful, but its utility is highly structured, sparse, failure-family-dependent, and concentrated in the immediately preceding event.** Treating “history” as one homogeneous feature block obscures this structure and can actively hurt some failure classes.

---

# 1. Baseline behavior and apparent trajectory effects

The notebook began from the existing semantic and transition classifiers.

### Baseline performance

| Model      |   Accuracy | Balanced Acc. |   Macro-F1 | Weighted-F1 |
| ---------- | ---------: | ------------: | ---------: | ----------: |
| Semantic   |     0.5252 |        0.4908 |     0.4974 |      0.5231 |
| Transition | **0.5353** |        0.4839 | **0.4981** |  **0.5323** |

Initially, comparing predictions gave:

* 69 apparent rescues
* 54 apparent breaks

For the 317 true `constraint_error` examples:

* both wrong: 170
* both correct: 114
* apparent rescues: 26
* apparent breaks: 7

At first glance this suggested that trajectory context was especially useful for constraint errors.

That interpretation turned out to be misleading.

---

# 2. Constraint-error rescue analysis

The notebook examined the 317 true constraint examples in detail.

The apparent rescue cases had dramatically different constraint probabilities from both-wrong examples.

For example:

| Feature                           | Rescue mean | Both-wrong mean | Cohen's d |
| --------------------------------- | ----------: | --------------: | --------: |
| transition constraint probability |      0.4312 |          0.1697 |  **2.47** |
| semantic constraint probability   |      0.3432 |          0.1450 |  **1.70** |
| constraint probability gain       |     +0.0880 |         +0.0247 |  **1.24** |

This established an important distinction.

Many apparent transition rescues were already **semantically easier constraint examples**. The transition model was not necessarily discovering a trajectory mechanism; it was operating on examples where the semantic model already assigned substantially more probability to the correct family.

The ranking analysis reinforced this. For rescues, the semantic model commonly placed the correct class around rank 2, while transition moved it to rank 1.

Thus:

> Transition frequently acted as a correction mechanism on near-miss semantic predictions rather than discovering completely hidden constraint failures.

---

# 3. Separating trajectory information from model/refitting effects

A major methodological correction in this notebook was distinguishing:

> **different predictions produced by a model containing trajectory features**

from

> **prediction changes actually caused by trajectory information.**

This mattered because independently retraining semantic and transition models changes:

* coefficients,
* regularization behavior,
* decision boundaries,
* probability calibration,

even when trajectory features contain no information for a particular example.

The notebook therefore constructed neutralized/zero-history counterfactuals.

This reduced the earlier apparent constraint effect dramatically.

Among 317 true constraints:

* original apparent rescues: **26**
* genuine trajectory rescues identified in the controlled comparison: **13**
* only **9 of the original 26 apparent rescues** were genuine under that attribution analysis
* 17 apparent rescues could not be attributed to trajectory features

This was an important correction to the earlier interpretation.

---

# 4. Zero-history sanity check

There were **56 zero-history examples**.

Every one of the 16 trajectory features was exactly zero on those examples.

This was verified directly:

* means = 0
* standard deviations = 0
* min/max = 0
* one unique value
* no trajectory feature varied without history

Furthermore, under direct history ablation:

* prediction changes: **0**
* zero-history trajectory rescues: **0**
* zero-history trajectory breaks: **0**

Therefore:

> Any difference between separately retrained semantic/trajectory classifiers on zero-history examples is a fitting/regularization effect, not evidence that historical context helped those examples.

This became an important methodological rule for the research.

---

# 5. Global causal effect of trajectory features

Using the controlled trajectory ablation, the notebook found:

### All examples

* trajectory rescues: **103**
* trajectory breaks: **79**
* net contribution: **+24**

So trajectory information has genuine positive aggregate utility.

But the average hides extreme differences across failure families.

### Failure-family effects

| True family           | Zero-history recall | Actual-history recall |    Δ recall | Rescues | Breaks |     Net |
| --------------------- | ------------------: | --------------------: | ----------: | ------: | -----: | ------: |
| workflow_error        |              0.5576 |            **0.6576** | **+0.1000** |      72 |      6 | **+66** |
| grounding_state_error |              0.3402 |            **0.4467** | **+0.1066** |      26 |      0 | **+26** |
| reasoning_value_error |              0.3871 |                0.4516 |     +0.0645 |       2 |      0 |      +2 |
| constraint_error      |          **0.5521** |                0.4416 | **−0.1104** |       3 |     38 | **−35** |
| tool_use_error        |          **0.5696** |                0.4219 | **−0.1477** |       0 |     35 | **−35** |

This was one of the notebook's most important discoveries.

> **Trajectory information is not globally beneficial.**

It strongly helps:

* workflow errors
* grounding-state errors

but strongly damages:

* constraint errors
* tool-use errors.

This immediately explains why a single global transition model produces only modest overall gains.

---

# 6. Trajectory changes the model's class prior / routing behavior

Full trajectory information changed **246 / 1489 predictions**, about **16.5%**.

Among those changes:

* 103 were rescues
* 79 were breaks
* 64 were wrong → wrong transitions

Trajectory therefore does much more than make occasional corrections.

It systematically redistributes predictions.

Prediction counts shifted strongly toward:

* `workflow_error`: **560 → 714**
* `grounding_state_error`: **171 → 226**

and away from:

* `constraint_error`: **420 → 325**
* `tool_use_error`: **315 → 199**

This is a crucial mechanistic observation:

> The transition representation has a directional bias: historical context pushes the classifier toward workflow/grounding interpretations and away from constraint/tool-use interpretations.

That bias is useful when correct and destructive when the underlying failure actually belongs to one of the displaced classes.

---

# 7. Continuous trajectory-strength experiment

Instead of simply switching history on/off, trajectory features were scaled by:

[
\alpha \in {0,;0.25,;0.50,;0.75,;1.0}
]

The implementation was validated:

* α=0 exactly matched zero-history probabilities.
* α=1 exactly matched actual-history probabilities.

### Overall results

|        α |   Accuracy | Balanced Acc. |   Macro-F1 | Changed vs zero |     Net |
| -------: | ---------: | ------------: | ---------: | --------------: | ------: |
|     0.00 |     0.5191 |        0.4813 |     0.4824 |               0 |       0 |
|     0.25 |     0.5245 |        0.4872 |     0.4905 |              63 |      +8 |
|     0.50 |     0.5299 |        0.4786 |     0.4871 |             122 |     +16 |
| **0.75** | **0.5366** |    **0.4875** |     0.4976 |             180 | **+26** |
|     1.00 |     0.5353 |        0.4839 | **0.4981** |             246 |     +24 |

Full trajectory strength was therefore not optimal.

A slightly damped representation around **α=0.75** produced the best accuracy and net rescue behavior.

This suggested:

> Trajectory evidence is useful, but the model tends to over-apply it.

---

# 8. Failure-family-specific optimal trajectory strength

The preferred trajectory strength depended dramatically on the baseline/zero-history predicted family.

Diagnostic choices were approximately:

* `workflow_error`: **0.25**
* `constraint_error`: **0.75**
* `tool_use_error`: **0.75**
* `grounding_state_error`: **1.0**
* `reasoning_value_error`: **0.0**

The diagnostic source-conditional policy achieved:

* accuracy **0.5400**
* balanced accuracy 0.4885
* weighted F1 **0.5389**
* 78 rescues
* 47 breaks
* **+31 net**

This was better than any single global α diagnostically.

However, cross-fitting destroyed most of that apparent gain.

### Cross-fitted adaptive alpha

* accuracy: **0.5232**
* balanced accuracy: **0.4705**
* macro-F1: **0.4824**
* weighted-F1: **0.5214**
* rescues: 64
* breaks: 58
* net: **+6**

The fold-selected α values were reasonably stable for some families—especially grounding, tool use, reasoning, and workflow—but constraint selection was unstable.

Therefore:

> Family-dependent trajectory utility is real, but the current dataset is too small/noisy for a learned per-family gating policy to generalize reliably.

This echoes the earlier MoE/router findings.

---

# 9. Which trajectory feature groups matter?

The original transition representation contained 16 trajectory features divided into:

* current cosine: 3
* current L1/L2 distance: 6
* history presence: 3
* history-transition cosine: 2
* history-transition distance: 2

Feature-group ablation produced a very useful result.

Removing `current_cosine` actually **improved** accuracy:

**0.5353 → 0.5413**

Removing `current_distance` reduced performance:

**0.5353 → 0.5339**

Zeroing trajectory dropped it further.

So the representation contained both useful and harmful historical signals.

---

# 10. Compact trajectory representations

The notebook then retrained models with different trajectory subsets.

The best representation was:

### `current_distance + presence`

9 features:

* accuracy **0.5420**
* balanced accuracy 0.4885
* macro-F1 **0.5028**
* weighted-F1 **0.5390**
* 46 rescues
* 27 breaks
* **+19 net**

Compare:

* zero trajectory: 0.5292
* all 16 trajectory features: 0.5353
* compact 9-feature representation: **0.5420**

Thus:

> More trajectory features are not better. A compact representation substantially outperformed the complete trajectory feature set.

`current_distance + history_transition` was similarly strong at 0.5413.

Cosine information was particularly suspicious: `all_except_current_cosine` also reached 0.5413.

---

# 11. L1/L2 and temporal-position experiments

The six current-distance features were decomposed further.

### Distance metric

L1-only and L2-only were almost identical:

* L1: accuracy 0.5386, macro-F1 0.5102
* L2: accuracy 0.5386, macro-F1 0.5103

Therefore there is little evidence that the exact distance norm is important.

### Temporal position

The more important result concerned **which previous event** was compared with the current event.

| Representation |   Accuracy | Balanced Acc. |   Macro-F1 |     Net |
| -------------- | ---------: | ------------: | ---------: | ------: |
| t−1 only       | **0.5393** |        0.4936 |     0.5070 |     +15 |
| t−2 only       |     0.5306 |        0.4905 |     0.5017 |      +2 |
| t−3 only       |     0.5306 |        0.4914 |     0.5019 |      +2 |
| t−1 + t−2      | **0.5393** |        0.4985 |     0.5113 | **+15** |
| t−1 + t−3      |     0.5386 |    **0.4998** | **0.5118** |     +14 |
| t−2 + t−3      |     0.5285 |        0.4822 |     0.4945 |  **−1** |

This produced a strong hypothesis:

> Most useful trajectory signal is concentrated in the immediately preceding event.

Older history becomes useful primarily in combination with the recent event, rather than independently.

---

# 12. Explicit temporal-dynamics features

The notebook then tested whether trajectory utility was really about *change over time* rather than raw distance.

Features included:

* (d_1)
* (d_1-d_2)
* (d_2-d_3)
* (d_1-d_3)
* (d_1/d_2)
* (d_1/d_3)

for both L1 and L2.

The best dynamic representation was `recent_plus_ratio`:

* accuracy **0.5400**
* balanced accuracy 0.4947
* macro-F1 0.5081
* net +16

But simple recent distance already achieved:

* 0.5393 accuracy
* +15 net

Difference-only and ratio-only representations were weak.

Therefore:

> The primary signal is the magnitude of recent displacement itself, not a sophisticated acceleration/trend statistic.

Dynamic ratios may add a tiny amount, but not enough to justify substantially greater complexity.

---

# 13. History-depth analysis

History availability was divided into:

* zero history: 56
* one event: 57
* two events: 63
* three or more: 1313

Among eligible examples, nested comparisons gave the cleanest evidence:

### t−1 versus semantic

Eligible n=1433:

* accuracy **0.5304 → 0.5387**
* 25 rescues
* 13 breaks
* **+12 net**

### Add t−2 after t−1

Eligible n=1376:

* 8 rescues
* 7 breaks
* **+1 net**

### Add t−3 after t−1

Eligible n=1313:

* 5 rescues
* 6 breaks
* **−1 net**

### t1+t3 versus t1+t2

* 8 rescues
* 11 breaks
* **−3 net**

This strongly reinforced:

> **t−1 captures most of the usable positional trajectory signal.**

The apparent zero-history improvement of separately trained distance models was explicitly recognized as a refitting effect and not interpreted as trajectory evidence.

---

# 14. Role-aware history

Because agent trajectories contain different event types, the notebook also investigated role-aware history.

There were **60 distinct recent role patterns**, with sequences dominated by combinations of `TOOL_CALL` and `ASSISTANT`.

Availability was high:

* previous assistant: ~86.3%
* previous tool: ~94.0%

Role-aware distances were constructed separately for previous assistant and tool events.

### Results

| Model               |   Accuracy | Balanced Acc. |   Macro-F1 | Net |
| ------------------- | ---------: | ------------: | ---------: | --: |
| semantic            |     0.5292 |        0.4777 |     0.4908 |   0 |
| positional distance | **0.5379** |    **0.4983** | **0.5105** | +13 |
| role-aware distance |     0.5359 |        0.4843 |     0.4989 | +10 |
| positional + role   |     0.5386 |        0.4852 |     0.5000 | +14 |

Role-aware history therefore did **not** beat simple positional distance.

But this should not be interpreted as evidence that event roles are irrelevant.

Rather, directly adding role-specific distances did not provide a better representation.

The later results suggest role may be more useful as a **conditioning variable describing when displacement should be trusted**, rather than as another block of distances.

---

# 15. Clean same-model lag intervention

The final experiment was the strongest mechanistic test in the notebook.

Instead of retraining models with different lag subsets, the full six-distance classifier was trained once. Then individual lags were zeroed at inference time.

This holds:

* semantic representation,
* model coefficients,
* regularization,
* scaler,
* training procedure

constant.

Only historical information changes.

### Results

| Intervention     |   Accuracy | Changed | Gains after removal | Losses after removal | Net removal |
| ---------------- | ---------: | ------: | ------------------: | -------------------: | ----------: |
| full distance    | **0.5379** |       0 |                   0 |                    0 |           0 |
| remove t−1       | **0.5151** |     181 |                  53 |               **87** |     **−34** |
| remove t−2       |     0.5393 |      91 |                  35 |                   33 |      **+2** |
| remove t−3       |     0.5319 |      69 |                  22 |                   31 |      **−9** |
| keep t−1 only    |     0.5353 |      75 |                  25 |                   29 |          −4 |
| zero all history |     0.5212 |     217 |                  68 |                   93 |     **−25** |

Eligible-only analysis confirmed it:

### t−1

n=1433:

**0.5373 → 0.5136**

Net removal: **−34**

### t−2

n=1376:

**0.5363 → 0.5378**

Net removal: **+2**

### t−3

n=1313:

**0.5423 → 0.5354**

Net removal: **−9**

This provides the clearest causal/mechanistic evidence in Notebook 15.

---

# Final conclusions of Notebook 15

## Conclusion 1 — trajectory context is real

Trajectory features are not merely exploiting retraining artifacts.

Controlled same-model interventions show that removing historical information causes genuine prediction degradation.

Zeroing all distance history produces **−25 net prediction utility** relative to the full model.

So trajectory modeling should **not** be abandoned.

---

## Conclusion 2 — trajectory utility is highly asymmetric by failure family

This is arguably the most important architectural result.

Trajectory strongly benefits:

**workflow errors**
→ +66 net

**grounding-state errors**
→ +26 net

while harming:

**constraint errors**
→ −35 net

**tool-use errors**
→ −35 net

Therefore a single homogeneous semantic+trajectory classifier is structurally mismatched to the problem.

The question is no longer:

> Should we use trajectory features?

It is:

> **Under what agent-state relationship should particular trajectory evidence affect a particular failure hypothesis?**

---

## Conclusion 3 — the previous event dominates

The cleanest result of the notebook is:

[
\boxed{t-1 \gg t-3 > t-2}
]

Removing t−1 causes **−34 net**.

Removing t−3 causes −9.

Removing t−2 is approximately neutral at +2.

Thus:

> **Agent reliability failures depend primarily on the relationship between the current event and the immediately preceding event, not on generic long-history similarity.**

---

## Conclusion 4 — older history is conditional, not additive

`t−3` contains some useful information, but its contribution interacts with the rest of history.

The lag effects are therefore not additive.

This argues against simply concatenating larger and larger history windows.

---

## Conclusion 5 — raw recent displacement beats complicated trajectory dynamics

Differences, ratios, history-transition features, and other engineered temporal dynamics provided little additional value over recent distance.

The signal appears to be simpler:

> **How different is the current state from the state immediately preceding it?**

What that displacement *means*, however, probably depends on the type of transition.

---

## Conclusion 6 — representation complexity is currently hurting

The full 16-feature trajectory representation was inferior to compact subsets.

* all trajectory: 0.5353 accuracy
* current distance + presence: **0.5420**
* removing current cosine: **0.5413**

Thus trajectory representation should become **smaller and more structured**, not larger.

---

## Conclusion 7 — routing remains attractive conceptually but learned routing is not yet reliable

Both the earlier selector/MoE research and this notebook point to the same pattern.

There is real conditional complementarity.

But attempting to learn a flexible router from the available data overfits.

The diagnostic adaptive α policy reached **0.5400**, whereas cross-fitting collapsed to **0.5232**.

So the evidence supports conditional routing, but **not yet a high-capacity learned router**.

---

# The emerging model of agent failure

Notebook 15 suggests a different conceptual architecture from the original generic semantic/trajectory classifier.

A failure should be viewed approximately as:

[
P(F_t \mid S_t,; E_{t-1}\rightarrow E_t,; \Delta_{t-1},; A_t)
]

where:

* (S_t) = semantic state of the current event;
* (E_{t-1}\rightarrow E_t) = type of agent transition;
* (\Delta_{t-1}) = recent semantic/state displacement;
* (A_t) = relevant agent action/state metadata.

The important object may therefore not be **trajectory history**.

It may be the **agent transition**.

For example:

`TOOL_CALL → TOOL_RESULT`

asks whether execution corresponded to the intended action.

`TOOL_RESULT → ASSISTANT`

asks whether the assistant correctly grounded its response in the observation.

`ASSISTANT → TOOL_CALL`

asks whether the next action follows correctly from reasoning/context.

`TOOL_CALL → TOOL_CALL`

may indicate correction, repeated attempts, multi-step execution, or pathological tool behavior.

`ASSISTANT → ASSISTANT`

has another semantic interpretation entirely.

The same numerical t−1 distance can mean very different things in each case.

---

# Recommended next research topic

## Notebook 16 — `16_event_transition_failure_mechanisms.ipynb`

I would **not** build another router yet.

I would investigate:

> ### **Event-transition-conditioned failure mechanisms**

The central research question should be:

> **When trajectory displacement helps or hurts, can that behavior be explained by the causal/structural relationship between the previous event and the current event?**

This is the missing link between the trajectory experiments and a reliable agent model.

### Phase 1: construct an agent event ontology

For every prediction point identify, where available:

* current event role/type;
* previous event role/type;
* previous assistant event;
* previous tool call;
* previous tool result;
* distance from current state to each;
* whether the immediately preceding event is assistant/tool/system/result/etc.;
* tool identity where relevant;
* whether current and previous events belong to the same action cycle.

Do **not** start with dozens of embedding features.

Start with structural variables.

### Phase 2: map t−1 utility by transition type

Reuse the clean fixed-model intervention:

**full distance vs remove-t−1**

Then compute for each transition type:

* support
* accuracy with t−1
* accuracy without t−1
* rescues
* breaks
* net utility
* true failure-family distribution

The key output should be:

[
U(t-1\mid\text{transition type},\text{failure family})
]

This tells us *where* recent context actually works.

### Phase 3: analyze error direction

The current notebook already shows systematic flows such as:

`constraint → workflow`

`tool_use → workflow`

and positive flows into grounding/workflow.

Notebook 16 should determine whether specific event transitions cause those flows.

For example, perhaps large displacement following a tool call is interpreted as workflow change even when the actual failure is incorrect tool usage.

That could explain the −35 tool-use trajectory effect.

### Phase 4: replace temporal lag with relational anchors

Only after the diagnostic analysis, test representations such as:

**current ↔ previous assistant**

**current ↔ triggering tool call**

**current ↔ tool result**

**tool call ↔ tool result**

**tool result ↔ subsequent assistant**

This is potentially much more meaningful than:

`current ↔ t−1`, `current ↔ t−2`, `current ↔ t−3`.

The hypothesis becomes:

> **Agent failures are relational failures between functional events, not generic temporal anomalies.**

### Phase 5: only then revisit routing

If Notebook 16 reveals stable mechanisms—for example:

* tool-result → assistant displacement predicts grounding failures;
* assistant → tool-call relations predict tool-use failures;
* recent displacement should be suppressed for constraint predictions;
* workflow failures benefit strongly from trajectory displacement—

then routing becomes much simpler.

Instead of learning an unrestricted MoE gate, we could construct a **small mechanism-aware reliability model**:

[
\text{semantic evidence}
+
\text{transition-specific evidence}
+
\text{small calibrated gate}
]

The gate would decide among a few **interpretable mechanisms**, rather than trying to infer from dozens of weak trajectory features whether the semantic or transition expert should win.

---

# Overall research trajectory

The notebooks now tell a coherent story:

**Earlier work:**
Semantic classification alone is limited.

↓

**Trajectory work:**
Historical context contains additional information.

↓

**Routing/MoE work:**
Semantic and trajectory models are complementary, but learned routing does not generalize robustly.

↓

**Notebook 14:**
Trajectory features contain measurable rescue signal, but generic learned selectors cannot reliably exploit it.

↓

**Notebook 15:**
The reason becomes clearer: trajectory utility is heterogeneous. It helps some failure families and hurts others; most useful information is concentrated at t−1; deeper history is largely redundant; and compact distance representations outperform richer ones.

↓

**Notebook 16:**
Determine **what t−1 actually represents structurally** by modeling event-to-event agent relationships.

↓

**Potential final architecture:**
A reliability classifier based on **semantic failure evidence + structured agent-transition evidence + conservative mechanism-aware routing/calibration**.

That is, in my view, the strongest path forward. The next gain is unlikely to come from another broad feature search or another larger classifier. The evidence now points toward discovering the **event-level causal structure of agent failures** and encoding that structure explicitly.
